# Full-Dataset Resegmentation + Cross-Platform QC (Xenium vs MERSCOPE)

This notebook runs **full-image (non-ROI)** Cellpose + ProSeg on:

- the Xenium dataset used in `XENIUM_Resegmentation_Tuning_ROI.ipynb`
- the MERSCOPE dataset used in `MERSCOPE_Resegmentation_Tuning_ROI.ipynb`

It then compares outputs with basic QC plots:

- cell geometry distributions (area, perimeter, eccentricity, circularity, solidity, aspect ratio)
- transcripts per cell and genes per cell (violin plots)
- assignment summary (total transcripts, assigned %, cells)

## Fixed params requested

- Cellpose: `flow_threshold=0.8`, `cellprob_threshold=-5.0`
- ProSeg: `base_proseg` settings from ROI notebook

## Notes

- This is a **heavy** run for whole images and can take a long time.
- Use run toggles (`RUN_MERSCOPE`, `RUN_XENIUM`) to execute one dataset at a time if needed.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys
import json
import time
import re
import shutil
import warnings
import gc
import ctypes
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.transform import resize

import spatialdata as sd
from spatialdata_io import xenium as xenium_reader
from cellpose import models
from tqdm.auto import tqdm

try:
    import psutil
except Exception:
    psutil = None

warnings.filterwarnings('ignore')
sns.set_context('notebook')
plt.rcParams['figure.figsize'] = (8, 5)

# robustly find repo root
repo = Path.cwd()
if not (repo / 'src' / 'resegmentation' / 'proseg_wrapper.py').exists():
    repo = repo.parent

sys.path.insert(0, str(repo / 'src' / 'resegmentation'))
from reseg_tools import filter_cell_by_regionprops as filter_masks_basic
from proseg_wrapper import run_proseg_refinement

print('repo:', repo)


In [ ]:
# -------------------------------
# Configuration
# -------------------------------
OUTPUT_ROOT = Path(os.environ.get(
    'MOSAIK_OUTPUT_ROOT',
    '/home/becalia/mnt/MOSAIK_analysis/tmp_full_reseg_compare_output'
))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Safety limits for this machine
MAX_SYSTEM_RAM_GB = 600.0
MEMORY_WARN_GB = 560.0
MEMORY_CHECK_EVERY_CHUNKS = 5

# Transcript chunking for lazy CSV generation
TRANSCRIPT_CHUNK_ROWS = 1_000_000
TRANSCRIPT_STATUS_EVERY_CHUNKS = 5

# Native-resolution Cellpose tiling (no rescaling)
CELLPOSE_TILE_SIZE_CANDIDATES = [6144, 4096, 3072, 2048]
CELLPOSE_TILE_OVERLAP = 256
CELLPOSE_MIN_TILE_SIZE = 1024
CELLPOSE_STATUS_EVERY_TILES = 10
CELLPOSE_FILTER_PER_TILE = True

# Full MERSCOPE dataset (from MERSCOPE tuning notebook context)
MERSCOPE_ZARR_PATH = Path('/media/mathieubo/SSD2/MerXen/P7513/MOSAIK_analysis/region_R2_mosaik_patched.zarr')
MERSCOPE_MANIFEST_PATH = MERSCOPE_ZARR_PATH / 'manifest.json'
MERSCOPE_TRANSFORM_PATH = MERSCOPE_ZARR_PATH / 'micron_to_mosaic_pixel_transform.csv'
MERSCOPE_IMAGE_KEY_PREFIX = '202509261108_P7513-2_VMSC19502_region_R2'
MERSCOPE_Z_RANGE = (0, 6)
MERSCOPE_CHANNELS = ['DAPI', 'PolyT']

# Full Xenium dataset (from Xenium tuning notebook context)
XENIUM_DIR_CANDIDATES = [
    Path('/media/mathieubo/SSD3/Xenium/20250917__131434__MB_RUN2_170925/output-XETG00443__0069511__P7513__20250917__131459/'),
    Path('/mnt/ISS2/Mathieu/Xenium/20250917__131434__MB_RUN2_170925/output-XETG00443__0069511__P7513__20250917__131459/'),
]
XENIUM_DIR = next((p for p in XENIUM_DIR_CANDIDATES if p.exists()), XENIUM_DIR_CANDIDATES[0])
XENIUM_SPEC_PATH = XENIUM_DIR / 'experiment.xenium'
XENIUM_CHANNELS = ['DAPI', '18S']
XENIUM_MIN_QV = 20.0

# Execution toggles
RUN_MERSCOPE = True
RUN_XENIUM = True
FORCE_RERUN = False

# Cellpose params requested
CELLPOSE_PARAMS = {
    'model_type': 'cyto3',
    'gpu': True,
    'diameter': None,
    'flow_threshold': 0.8,
    'cellprob_threshold': -5.0,
    'tile_overlap': 0.15,
    'bsize': 256,
    'factor_rescale': 1.0,  # locked to native resolution per user request
}

# Mask post-filtering (same defaults as ROI notebook)
MASK_FILTER_PARAMS = {
    'max_eccentricity': 0.99,
    'min_area_percentile': 1.0,
    'min_area_px': None,
    'n_jobs': 16,
    'show_progress': True,
}

# ProSeg base params requested (from ROI notebook)
BASE_PROSEG_PARAMS = {
    'samples': 1200,
    'voxel_size': 0.5,
    'burnin_voxel_size': 1.0,
    'nuclear_reassignment_prob': 0.25,
    'diffusion_probability': 0.25,
    'cell_compactness': 0.04,
    'expand_initialized_cells': 0,
    'prior_seg_reassignment_prob': 0.5,
    'use_cell_initialization': True,
    'max_transcript_nucleus_distance': 60.0,
    'diffusion_sigma_far': None,
    'num_threads': 64,
}

# Dataset-specific override for voxel layers
DATASET_PROSEG_OVERRIDES = {
    'MERSCOPE': {'voxel_layers': MERSCOPE_Z_RANGE[1] - MERSCOPE_Z_RANGE[0] + 1},
    'XENIUM': {'voxel_layers': 2},
}

PROSEG_BINARY = Path('/home/becalia/.cargo/bin/proseg')

print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('MERSCOPE_ZARR_PATH exists:', MERSCOPE_ZARR_PATH.exists())
print('XENIUM_DIR exists:', XENIUM_DIR.exists())
print('PROSEG_BINARY exists:', PROSEG_BINARY.exists())





In [ ]:
# -------------------------------
# Helpers
# -------------------------------
def _now():
    return datetime.now().strftime('%H:%M:%S')


def memory_snapshot_gb():
    if psutil is None:
        return {'rss_gb': np.nan, 'used_gb': np.nan, 'available_gb': np.nan}
    vm = psutil.virtual_memory()
    rss = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
    return {
        'rss_gb': rss,
        'used_gb': vm.used / (1024 ** 3),
        'available_gb': vm.available / (1024 ** 3),
    }


def status(msg):
    mem = memory_snapshot_gb()
    print(
        f"[{_now()}] {msg} | "
        f"RSS={mem['rss_gb']:.1f} GB | "
        f"System used={mem['used_gb']:.1f} GB | "
        f"System avail={mem['available_gb']:.1f} GB"
    )


def enforce_memory_limit(stage=''):
    if psutil is None:
        return
    used_gb = psutil.virtual_memory().used / (1024 ** 3)
    if used_gb > MAX_SYSTEM_RAM_GB:
        raise MemoryError(
            f"System RAM usage exceeded limit at '{stage}': "
            f"{used_gb:.1f} GB > {MAX_SYSTEM_RAM_GB:.1f} GB"
        )
    if used_gb > MEMORY_WARN_GB:
        status(
            f"WARNING: High RAM usage during '{stage}' "
            f"({used_gb:.1f} GB > warn {MEMORY_WARN_GB:.1f} GB)"
        )


def force_release(note=''):
    gc.collect()
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass
    if note:
        status(f"Memory cleanup complete: {note}")


def clear_cuda_cache():
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass
    except Exception:
        pass


def _to_pandas(df_like):
    if hasattr(df_like, 'compute') and not isinstance(df_like, pd.DataFrame):
        return df_like.compute()
    return df_like.copy()


def _resolve_col(df_like, candidates, required=True):
    cols = set(map(str, list(df_like.columns)))
    for c in candidates:
        if c in cols:
            return c
    if required:
        raise KeyError(f'Could not resolve required column from candidates={candidates}. Available={list(df_like.columns)}')
    return None


def _ensure_no_rescale():
    scale = float(CELLPOSE_PARAMS.get('factor_rescale', 1.0))
    if abs(scale - 1.0) > 1e-12:
        raise ValueError(
            'This notebook is configured for native-resolution tiled Cellpose only. '
            f'Got factor_rescale={scale}. Please set CELLPOSE_PARAMS["factor_rescale"] = 1.0.'
        )


def list_plane_keys(images, prefix=None):
    pat = re.compile(r'^(?P<prefix>.+)_z(?P<z>\d+)$')
    out = []
    for k in images.keys():
        m = pat.match(str(k))
        if not m:
            continue
        if prefix is not None and not str(k).startswith(prefix):
            continue
        out.append((int(m.group('z')), str(k)))
    return sorted(out)


def _get_image_dataarray(img_obj):
    img_xr = img_obj
    if hasattr(img_obj, '__contains__') and 'scale0' in img_obj:
        img_xr = img_obj['scale0'].ds['image']
    else:
        try:
            img_xr = img_obj['scale0'].ds['image']
        except Exception:
            pass
    return img_xr


def build_image_source(img_obj, requested_channels=None, as_float32=True):
    img_xr = _get_image_dataarray(img_obj)

    if hasattr(img_xr, 'dims'):
        if all(d in img_xr.dims for d in ('y', 'x', 'c')):
            da = img_xr.transpose('y', 'x', 'c')
        elif all(d in img_xr.dims for d in ('c', 'y', 'x')):
            da = img_xr.transpose('y', 'x', 'c')
        elif all(d in img_xr.dims for d in ('y', 'x')):
            da = img_xr.expand_dims(c=['c0']).transpose('y', 'x', 'c')
        else:
            da = None

        if da is not None:
            channels = [str(c) for c in da.coords['c'].values] if 'c' in da.coords else [f'c{i}' for i in range(da.shape[-1])]
            if requested_channels is not None and len(channels) > 0:
                keep = [c for c in requested_channels if c in channels]
                if not keep:
                    raise ValueError(f"Requested channels {requested_channels} not found. Available channels: {channels}")
                idx = [channels.index(c) for c in keep]
                da = da.isel(c=idx)
                channels = keep

            h = int(da.sizes['y'])
            w = int(da.sizes['x'])
            c = int(da.sizes['c'])
            return {
                'kind': 'xarray',
                'data': da,
                'channels': channels,
                'shape': (h, w, c),
                'as_float32': as_float32,
            }

    arr = img_xr.data.compute() if hasattr(img_xr, 'data') else np.asarray(img_xr)
    if arr.ndim == 2:
        arr = arr[..., np.newaxis]
    elif arr.ndim == 3 and arr.shape[0] <= 8 and arr.shape[-1] > 8:
        arr = np.moveaxis(arr, 0, -1)

    channels = [f'c{i}' for i in range(arr.shape[-1])]
    if requested_channels is not None and len(channels) > 0:
        keep = [c for c in requested_channels if c in channels]
        if not keep:
            raise ValueError(f"Requested channels {requested_channels} not found. Available channels: {channels}")
        idx = [channels.index(c) for c in keep]
        arr = arr[..., idx]
        channels = keep

    if as_float32:
        arr = arr.astype(np.float32, copy=False)

    return {
        'kind': 'array',
        'data': arr,
        'channels': channels,
        'shape': arr.shape,
        'as_float32': as_float32,
    }


def fetch_tile(source, y0, y1, x0, x1):
    if source['kind'] == 'xarray':
        arr = source['data'].isel(y=slice(y0, y1), x=slice(x0, x1)).data.compute()
    else:
        arr = source['data'][y0:y1, x0:x1, :]
    if source.get('as_float32', False):
        arr = arr.astype(np.float32, copy=False)
    return arr


def prepare_merscope_plane_sources(sdata, selected_keys, requested_channels=None):
    if len(selected_keys) == 0:
        raise ValueError('selected_keys is empty')

    sources = []
    for k in selected_keys:
        sources.append(build_image_source(sdata.images[k], requested_channels=requested_channels, as_float32=True))

    base_shape = sources[0]['shape']
    for src in sources[1:]:
        if src['shape'][:2] != base_shape[:2]:
            raise ValueError(f'Inconsistent plane shapes: {base_shape[:2]} vs {src["shape"][:2]}')

    h, w, _ = base_shape
    channels = sources[0]['channels']
    return sources, h, w, channels


def fetch_merscope_projected_tile(plane_sources, y0, y1, x0, x1):
    proj = None
    for src in plane_sources:
        arr = fetch_tile(src, y0, y1, x0, x1)
        if proj is None:
            proj = arr
        else:
            np.maximum(proj, arr, out=proj)
        del arr
    return proj


def prepare_cellpose_input(image, factor_rescale=1.0):
    img = image.astype(np.float32, copy=False)
    p2, p98 = np.percentile(img, (2, 98))
    img = np.clip((img - p2) / (p98 - p2 + 1e-8), 0, 1)
    img8 = (img * 255).astype(np.uint8)

    if img8.ndim == 2:
        img8 = np.stack([img8] * 3, axis=-1)
    elif img8.shape[-1] == 1:
        img8 = np.repeat(img8, 3, axis=-1)
    elif img8.shape[-1] == 2:
        img8 = np.concatenate([img8, np.zeros_like(img8[..., :1])], axis=-1)
    elif img8.shape[-1] > 3:
        img8 = img8[..., :3]

    scale_factor = float(factor_rescale)
    if scale_factor > 1.0:
        raise ValueError('factor_rescale > 1.0 is disabled in this native-resolution notebook.')

    return img8, img8, 1.0


def build_cellpose_model(cp_params):
    kwargs = {'gpu': cp_params.get('gpu', True)}
    if cp_params.get('model_type') is not None:
        kwargs['model_type'] = cp_params['model_type']
    return models.CellposeModel(**kwargs)


def run_cellpose_model_eval(model, img_seg, cp_params):
    return model.eval(
        img_seg,
        diameter=cp_params.get('diameter', None),
        flow_threshold=cp_params.get('flow_threshold', 0.4),
        cellprob_threshold=cp_params.get('cellprob_threshold', 0.0),
        tile_overlap=cp_params.get('tile_overlap', 0.15),
        bsize=cp_params.get('bsize', 256),
    )


def iter_core_tiles(height, width, tile_size, overlap):
    tile_size = int(tile_size)
    overlap = int(overlap)
    core = tile_size - (2 * overlap)
    if core <= 0:
        raise ValueError(f'Invalid tile config: tile_size={tile_size}, overlap={overlap}')

    ys = list(range(0, int(height), core))
    xs = list(range(0, int(width), core))

    for y_core0 in ys:
        y_core1 = min(y_core0 + core, int(height))
        y_tile0 = max(0, y_core0 - overlap)
        y_tile1 = min(int(height), y_core1 + overlap)
        y_core0_in_tile = y_core0 - y_tile0
        y_core1_in_tile = y_core0_in_tile + (y_core1 - y_core0)

        for x_core0 in xs:
            x_core1 = min(x_core0 + core, int(width))
            x_tile0 = max(0, x_core0 - overlap)
            x_tile1 = min(int(width), x_core1 + overlap)
            x_core0_in_tile = x_core0 - x_tile0
            x_core1_in_tile = x_core0_in_tile + (x_core1 - x_core0)

            yield {
                'core_y0': y_core0,
                'core_y1': y_core1,
                'core_x0': x_core0,
                'core_x1': x_core1,
                'tile_y0': y_tile0,
                'tile_y1': y_tile1,
                'tile_x0': x_tile0,
                'tile_x1': x_tile1,
                'core_y0_in_tile': y_core0_in_tile,
                'core_y1_in_tile': y_core1_in_tile,
                'core_x0_in_tile': x_core0_in_tile,
                'core_x1_in_tile': x_core1_in_tile,
            }


def choose_working_tile_size(fetch_tile_fn, height, width, model, cp_params, candidates, dataset_name):
    cy = int(height) // 2
    cx = int(width) // 2

    for ts in candidates:
        ts = int(ts)
        y0 = max(0, cy - ts // 2)
        x0 = max(0, cx - ts // 2)
        y1 = min(int(height), y0 + ts)
        x1 = min(int(width), x0 + ts)
        y0 = max(0, y1 - ts)
        x0 = max(0, x1 - ts)

        status(f"[{dataset_name}] Probing Cellpose tile size={ts} on center tile")
        try:
            tile = fetch_tile_fn(y0, y1, x0, x1)
            _, img_seg, _ = prepare_cellpose_input(tile, factor_rescale=1.0)
            masks, flows, styles = run_cellpose_model_eval(model, img_seg, cp_params)
            status(f"[{dataset_name}] Tile size {ts} succeeded (labels in probe={int(np.max(masks))})")
            del tile, img_seg, masks, flows, styles
            clear_cuda_cache()
            force_release()
            return ts
        except (RuntimeError, MemoryError) as e:
            msg = str(e).lower()
            if ('out of memory' in msg) or ('cuda' in msg and 'memory' in msg) or isinstance(e, MemoryError):
                status(f"[{dataset_name}] Tile size {ts} failed due to memory; trying smaller tile")
                clear_cuda_cache()
                force_release()
                continue
            raise

    raise RuntimeError(f"[{dataset_name}] Could not find a working tile size from candidates={list(candidates)}")


def relabel_core_to_global(core_mask, next_label):
    core_mask = core_mask.astype(np.int64, copy=False)
    labels = np.unique(core_mask)
    labels = labels[labels > 0]

    if labels.size == 0:
        return np.zeros(core_mask.shape, dtype=np.uint32), int(next_label), 0

    max_label = int(labels.max())
    new_ids = np.arange(int(next_label), int(next_label) + int(labels.size), dtype=np.uint32)

    if max_label < 20_000_000:
        lut = np.zeros(max_label + 1, dtype=np.uint32)
        lut[labels] = new_ids
        core_global = lut[core_mask]
    else:
        core_global = np.zeros(core_mask.shape, dtype=np.uint32)
        for old, new in zip(labels, new_ids):
            core_global[core_mask == old] = new

    return core_global, int(next_label) + int(labels.size), int(labels.size)


def run_tiled_cellpose_segmentation(
    fetch_tile_fn,
    height,
    width,
    dataset_name,
    output_mask_path,
    cp_params,
    mask_filter_params,
    tile_size_candidates,
    overlap,
    min_tile_size,
    status_every_tiles=10,
    apply_filter_per_tile=True,
):
    output_mask_path = Path(output_mask_path)
    if output_mask_path.exists():
        output_mask_path.unlink()

    candidates = [int(x) for x in tile_size_candidates if int(x) >= int(min_tile_size)]
    if not candidates:
        raise ValueError('No valid tile sizes in tile_size_candidates after min_tile_size filter')

    model = build_cellpose_model(cp_params)
    tile_size = choose_working_tile_size(
        fetch_tile_fn=fetch_tile_fn,
        height=height,
        width=width,
        model=model,
        cp_params=cp_params,
        candidates=candidates,
        dataset_name=dataset_name,
    )

    tiles = list(iter_core_tiles(height=height, width=width, tile_size=tile_size, overlap=overlap))
    status(f"[{dataset_name}] Running tiled Cellpose over {len(tiles)} tiles (tile_size={tile_size}, overlap={overlap})")

    mask_mem = np.lib.format.open_memmap(
        str(output_mask_path),
        mode='w+',
        dtype=np.uint32,
        shape=(int(height), int(width)),
    )
    mask_mem[:] = 0

    next_label = 1
    total_new_labels = 0
    pbar = tqdm(tiles, total=len(tiles), desc=f'[{dataset_name}] Cellpose tiles', unit='tile')

    # Avoid spawning too many process pools repeatedly for small tiles.
    tile_filter_n_jobs = max(1, min(int(mask_filter_params.get('n_jobs', 1)), 4))

    for i, t in enumerate(pbar, start=1):
        enforce_memory_limit(stage=f'{dataset_name} tile {i}/{len(tiles)}')

        tile = fetch_tile_fn(t['tile_y0'], t['tile_y1'], t['tile_x0'], t['tile_x1'])
        _, img_seg, _ = prepare_cellpose_input(tile, factor_rescale=1.0)

        masks, flows, styles = run_cellpose_model_eval(model, img_seg, cp_params)

        if apply_filter_per_tile:
            masks = filter_masks_basic(
                masks,
                max_eccentricity=mask_filter_params['max_eccentricity'],
                n_jobs=tile_filter_n_jobs,
                show_progress=False,
                min_area_percentile=mask_filter_params['min_area_percentile'],
                min_area_px=mask_filter_params['min_area_px'],
            )

        core = masks[
            t['core_y0_in_tile']:t['core_y1_in_tile'],
            t['core_x0_in_tile']:t['core_x1_in_tile'],
        ]

        core_global, next_label, n_new = relabel_core_to_global(core, next_label=next_label)
        total_new_labels += n_new

        mask_mem[t['core_y0']:t['core_y1'], t['core_x0']:t['core_x1']] = core_global

        if psutil is not None:
            pbar.set_postfix_str(
                f"labels={total_new_labels:,} rss={memory_snapshot_gb()['rss_gb']:.1f}GB"
            )

        if i % int(status_every_tiles) == 0:
            status(
                f"[{dataset_name}] tile {i}/{len(tiles)} complete; "
                f"accumulated labels={total_new_labels:,}"
            )

        del tile, img_seg, masks, flows, styles, core, core_global
        clear_cuda_cache()

        if i % int(MEMORY_CHECK_EVERY_CHUNKS) == 0:
            force_release()

    mask_mem.flush()
    del mask_mem, model
    clear_cuda_cache()
    force_release(note=f'after tiled Cellpose {dataset_name}')

    status(f"[{dataset_name}] Tiled Cellpose complete; wrote masks to {output_mask_path}")
    return output_mask_path


def build_cellpose_affine_to_microns(M, scale_factor, x0=0.0, y0=0.0):
    Minv = np.linalg.inv(M)
    S = np.array([
        [scale_factor, 0.0, float(x0)],
        [0.0, scale_factor, float(y0)],
        [0.0, 0.0, 1.0],
    ], dtype=float)
    T = Minv @ S
    x_transform = (float(T[0, 0]), float(T[0, 1]), float(T[0, 2]))
    y_transform = (float(T[1, 0]), float(T[1, 1]), float(T[1, 2]))
    return x_transform, y_transform


def _invert_mask_affine(x_transform, y_transform):
    A = np.array([
        [float(x_transform[0]), float(x_transform[1])],
        [float(y_transform[0]), float(y_transform[1])],
    ], dtype=float)
    b = np.array([float(x_transform[2]), float(y_transform[2])], dtype=float)
    A_inv = np.linalg.inv(A)
    return A_inv, b


def assign_labels_from_masks(x_micron, y_micron, masks, A_inv, b):
    coords = np.vstack([x_micron, y_micron])
    pix = A_inv @ (coords - b[:, None])

    x_px = np.rint(pix[0]).astype(np.int64)
    y_px = np.rint(pix[1]).astype(np.int64)

    h, w = masks.shape
    valid = (x_px >= 0) & (x_px < w) & (y_px >= 0) & (y_px < h)

    labels = np.zeros(len(x_micron), dtype=np.int32)
    labels[valid] = masks[y_px[valid], x_px[valid]].astype(np.int32, copy=False)
    return labels


def _iter_points_chunks(points_obj, columns, chunk_rows=1_000_000, desc='Transcript chunks'):
    if hasattr(points_obj, 'npartitions') and hasattr(points_obj, 'partitions'):
        n_parts = int(points_obj.npartitions)
        pbar = tqdm(range(n_parts), total=n_parts, desc=desc, unit='part')
        for pidx in pbar:
            part = points_obj.partitions[pidx][columns].compute()
            yield part
        return

    if hasattr(points_obj, 'compute') and not isinstance(points_obj, pd.DataFrame):
        status('[Points] Materializing non-partitioned lazy points object (fallback path)')
        points_obj = points_obj[columns].compute()

    n_rows = len(points_obj)
    n_chunks = (n_rows + chunk_rows - 1) // chunk_rows
    pbar = tqdm(range(0, n_rows, chunk_rows), total=n_chunks, desc=desc, unit='chunk')
    for start in pbar:
        stop = min(start + chunk_rows, n_rows)
        yield points_obj.iloc[start:stop][columns].copy()


def write_proseg_csv_from_points(
    points_obj,
    csv_path,
    masks,
    x_transform,
    y_transform,
    x_col,
    y_col,
    z_col,
    gene_col,
    qv_col=None,
    min_qv=None,
    chunk_rows=1_000_000,
    dataset_name='DATASET',
    status_every_chunks=5,
):
    csv_path = Path(csv_path)
    if csv_path.exists():
        csv_path.unlink()

    A_inv, b = _invert_mask_affine(x_transform, y_transform)

    cols = [x_col, y_col, gene_col]
    if z_col is not None:
        cols.append(z_col)
    if qv_col is not None and qv_col not in cols:
        cols.append(qv_col)

    n_input = 0
    n_written = 0
    n_seeded = 0
    header_written = False

    status(f"[{dataset_name}] Writing ProSeg CSV lazily to: {csv_path}")
    chunk_iter = _iter_points_chunks(
        points_obj,
        columns=cols,
        chunk_rows=chunk_rows,
        desc=f'[{dataset_name}] transcript chunks',
    )

    for i, chunk in enumerate(chunk_iter, start=1):
        n_input += len(chunk)

        x_vals = pd.to_numeric(chunk[x_col], errors='coerce').to_numpy(np.float64)
        y_vals = pd.to_numeric(chunk[y_col], errors='coerce').to_numpy(np.float64)

        if z_col is None:
            z_vals = np.zeros(len(chunk), dtype=np.float32)
        else:
            z_vals = pd.to_numeric(chunk[z_col], errors='coerce').fillna(0.0).to_numpy(np.float32)

        gene_vals = chunk[gene_col].astype(str).to_numpy(dtype=object)

        valid = np.isfinite(x_vals) & np.isfinite(y_vals)
        valid &= pd.notna(gene_vals)
        valid &= (gene_vals != '')

        if qv_col is not None and min_qv is not None:
            qv_vals = pd.to_numeric(chunk[qv_col], errors='coerce').to_numpy(np.float64)
            valid &= np.isfinite(qv_vals) & (qv_vals >= float(min_qv))

        if np.any(valid):
            xv = x_vals[valid]
            yv = y_vals[valid]
            zv = z_vals[valid]
            gv = gene_vals[valid]

            labels = assign_labels_from_masks(xv, yv, masks, A_inv=A_inv, b=b)

            out_df = pd.DataFrame({
                'x_micron': xv.astype(np.float32, copy=False),
                'y_micron': yv.astype(np.float32, copy=False),
                'z_micron': zv.astype(np.float32, copy=False),
                'feature_name': gv.astype(str),
                'cell_id': labels.astype(np.int32, copy=False),
            })

            out_df.to_csv(
                csv_path,
                mode='w' if not header_written else 'a',
                header=not header_written,
                index=False,
            )

            header_written = True
            n_written += len(out_df)
            n_seeded += int((labels != 0).sum())

            del xv, yv, zv, gv, labels, out_df

        if i % status_every_chunks == 0:
            status(
                f"[{dataset_name}] chunk={i} input={n_input:,} "
                f"written={n_written:,} seeded={n_seeded:,}"
            )

        if i % MEMORY_CHECK_EVERY_CHUNKS == 0:
            enforce_memory_limit(stage=f'{dataset_name} transcript chunk {i}')

        del chunk, x_vals, y_vals, z_vals, gene_vals, valid

    if not header_written:
        raise RuntimeError(
            f"[{dataset_name}] No transcripts were written to {csv_path}. "
            "Check coordinate/qv filters and selected columns."
        )

    pct_seeded = 100.0 * n_seeded / max(n_written, 1)
    status(
        f"[{dataset_name}] ProSeg CSV complete: {n_written:,} rows written, "
        f"{n_seeded:,} seeded ({pct_seeded:.2f}%), input={n_input:,}"
    )

    return {
        'csv_path': csv_path,
        'n_input': int(n_input),
        'n_written': int(n_written),
        'n_seeded': int(n_seeded),
        'pct_seeded': float(pct_seeded),
    }



def _normalize_points_for_latest_write(points_obj, points_key='points'):
    """Normalize point-table dtypes so all Dask partitions share one pyarrow schema."""
    if not hasattr(points_obj, 'columns'):
        return points_obj

    df = points_obj
    cols = set(map(str, list(df.columns)))

    # Gene dictionaries can exceed int8 code range across partitions; force plain string.
    for gene_col in ('gene', 'feature_name', 'target'):
        if gene_col in cols:
            try:
                df[gene_col] = df[gene_col].astype('string')
            except Exception:
                df[gene_col] = df[gene_col].astype(str)

    # These identifiers can arrive mixed (uint/float/string) across partitions.
    # Prefer unsigned ints; fall back to string if coercion is unsafe.
    int_casts = {
        'transcript_id': 'uint64',
        'assignment': 'uint32',
        'cell': 'uint32',
    }
    for c, target_dtype in int_casts.items():
        if c in cols:
            try:
                df[c] = df[c].astype('float64').fillna(0).astype(target_dtype)
            except Exception:
                try:
                    df[c] = df[c].fillna(0).astype(target_dtype)
                except Exception:
                    df[c] = df[c].astype('string')

    # Keep cell_id as string-like for compatibility with downstream assignment logic.
    if 'cell_id' in cols:
        try:
            df['cell_id'] = df['cell_id'].astype('string')
        except Exception:
            df['cell_id'] = df['cell_id'].astype(str)

    status(f"Normalized points schema for '{points_key}' (columns={len(cols)})")
    return df


def convert_to_latest_zarr(raw_path, latest_path):
    raw_path = Path(raw_path)
    latest_path = Path(latest_path)

    status(f"Converting to latest SpatialData layout: {raw_path} -> {latest_path}")
    if latest_path.exists():
        shutil.rmtree(latest_path)

    sdata = sd.read_zarr(raw_path)

    # Normalize point-table schemas before write to avoid pyarrow partition mismatch.
    for points_key in list(sdata.points.keys()):
        try:
            sdata.points[points_key] = _normalize_points_for_latest_write(
                sdata.points[points_key], points_key=points_key
            )
        except Exception as e:
            status(
                f"WARNING: Could not normalize points '{points_key}' before latest write: {e}"
            )

    sdata.write(latest_path)

    del sdata
    force_release(note='after latest SpatialData write')
    return latest_path


def assignment_mask(series):
    if pd.api.types.is_numeric_dtype(series):
        vals = pd.to_numeric(series, errors='coerce').fillna(0)
        return vals != 0
    s = series.astype(str).str.strip().str.lower()
    return (~s.isin({'0', '0.0', '', 'none', 'nan', 'null', 'unassigned'})) & series.notna()


def first_existing_col(df_like, cols):
    all_cols = set(map(str, list(df_like.columns)))
    for c in cols:
        if c in all_cols:
            return c
    return None


def _primary_polygon(geom):
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == 'Polygon':
        return geom
    if geom.geom_type == 'MultiPolygon':
        return max(geom.geoms, key=lambda g: g.area)
    return None


def _eccentricity_aspect(geom):
    poly = _primary_polygon(geom)
    if poly is None:
        return np.nan, np.nan
    coords = np.asarray(poly.exterior.coords)
    if coords.shape[0] < 5:
        return np.nan, np.nan
    xy = coords[:, :2]
    xy = xy - xy.mean(axis=0, keepdims=True)
    cov = np.cov(xy, rowvar=False)
    eigvals = np.linalg.eigvalsh(cov)
    eigvals = np.sort(np.clip(eigvals, 1e-12, None))
    major = float(np.sqrt(eigvals[1]))
    minor = float(np.sqrt(eigvals[0]))
    if major <= 0:
        return np.nan, np.nan
    ecc = float(np.sqrt(max(0.0, 1.0 - (minor ** 2) / (major ** 2))))
    aspect = float(major / minor) if minor > 0 else np.nan
    return ecc, aspect


def _compute_cell_metrics_from_points(points_obj, assign_col, gene_col):
    if hasattr(points_obj, 'npartitions') and hasattr(points_obj, 'partitions'):
        try:
            pts_small = points_obj[[assign_col, gene_col]]
            n_total = int(pts_small.shape[0].compute())

            assigned_mask_dd = pts_small[assign_col].map_partitions(
                assignment_mask,
                meta=(assign_col, 'bool'),
            )
            n_assigned = int(assigned_mask_dd.sum().compute())

            assigned_dd = pts_small[assigned_mask_dd]
            assigned_dd = assigned_dd.assign(cell_id_norm=assigned_dd[assign_col].astype(str))

            trans_per_cell = assigned_dd.groupby('cell_id_norm').size().compute().rename('transcripts_per_cell')
            genes_per_cell = assigned_dd.groupby('cell_id_norm')[gene_col].nunique().compute().rename('genes_per_cell')

            cell_metrics = pd.concat([trans_per_cell, genes_per_cell], axis=1).reset_index(drop=False)
            return n_total, n_assigned, cell_metrics
        except Exception as exc:
            status(f"Dask QC aggregation failed ({exc}); falling back to pandas compute().")

    pts = _to_pandas(points_obj)

    is_assigned = assignment_mask(pts[assign_col])
    n_total = int(len(pts))
    n_assigned = int(is_assigned.sum())

    assigned_pts = pts.loc[is_assigned, [assign_col, gene_col]].copy()
    assigned_pts['cell_id_norm'] = assigned_pts[assign_col].astype(str)

    trans_per_cell = assigned_pts.groupby('cell_id_norm').size().rename('transcripts_per_cell')
    genes_per_cell = assigned_pts.groupby('cell_id_norm')[gene_col].nunique().rename('genes_per_cell')
    cell_metrics = pd.concat([trans_per_cell, genes_per_cell], axis=1).reset_index(drop=False)
    return n_total, n_assigned, cell_metrics


def compute_dataset_qc(latest_zarr_path, dataset_name):
    status(f"[{dataset_name}] Loading latest output for QC: {latest_zarr_path}")
    sdata = sd.read_zarr(latest_zarr_path)

    if len(sdata.shapes) == 0:
        raise RuntimeError(f'No shapes found in {latest_zarr_path}')

    shape_key = list(sdata.shapes.keys())[0]
    shapes = sdata.shapes[shape_key]
    gdf = shapes[['geometry']].copy() if 'geometry' in shapes.columns else gpd.GeoDataFrame({'geometry': shapes.geometry})
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

    geom_df = pd.DataFrame(index=gdf.index)
    geom_df['dataset'] = dataset_name
    geom_df['area'] = gdf.geometry.area.values
    geom_df['perimeter'] = gdf.geometry.length.values
    geom_df['convex_area'] = gdf.geometry.convex_hull.area.values
    geom_df['circularity'] = 4.0 * np.pi * geom_df['area'] / np.clip(geom_df['perimeter'] ** 2, 1e-12, None)
    geom_df['solidity'] = geom_df['area'] / np.clip(geom_df['convex_area'], 1e-12, None)
    ea = gdf.geometry.apply(_eccentricity_aspect)
    geom_df['eccentricity'] = [x[0] for x in ea]
    geom_df['aspect_ratio'] = [x[1] for x in ea]
    geom_df['log10_area'] = np.log10(np.clip(geom_df['area'].values, 1e-9, None))

    if len(sdata.points) == 0:
        raise RuntimeError(f'No points found in {latest_zarr_path}')

    points_key = list(sdata.points.keys())[0]
    pts_obj = sdata.points[points_key]

    assign_col = first_existing_col(pts_obj, ['assignment', 'cell', 'cell_id'])
    gene_col = first_existing_col(pts_obj, ['feature_name', 'gene', 'target'])
    if assign_col is None:
        raise KeyError(f'No assignment column found in points columns={list(pts_obj.columns)}')
    if gene_col is None:
        raise KeyError(f'No gene column found in points columns={list(pts_obj.columns)}')

    n_total, n_assigned, cell_metrics = _compute_cell_metrics_from_points(pts_obj, assign_col=assign_col, gene_col=gene_col)
    cell_metrics['dataset'] = dataset_name

    pct_assigned = 100.0 * n_assigned / max(n_total, 1)

    summary = {
        'dataset': dataset_name,
        'latest_zarr_path': str(latest_zarr_path),
        'n_cells': int(len(gdf)),
        'n_transcripts_total': int(n_total),
        'n_transcripts_assigned': int(n_assigned),
        'pct_assigned': pct_assigned,
        'median_area': float(np.nanmedian(geom_df['area'])),
        'median_eccentricity': float(np.nanmedian(geom_df['eccentricity'])),
        'median_transcripts_per_cell': float(np.nanmedian(cell_metrics['transcripts_per_cell'])) if len(cell_metrics) else np.nan,
        'median_genes_per_cell': float(np.nanmedian(cell_metrics['genes_per_cell'])) if len(cell_metrics) else np.nan,
    }

    del sdata, gdf, shapes
    force_release(note=f'after QC {dataset_name}')

    return {
        'summary': summary,
        'geometry_metrics': geom_df,
        'cell_metrics': cell_metrics,
    }


In [ ]:
def run_full_merscope(output_dir):
    _ensure_no_rescale()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    raw_output = output_dir / 'proseg_base_raw.zarr'
    latest_output = output_dir / 'proseg_base_latest.zarr'
    transcripts_csv = output_dir / 'transcripts_for_proseg.csv'
    mask_path = output_dir / 'cellpose_masks_tiled.npy'

    if latest_output.exists() and (not FORCE_RERUN):
        status(f"[MERSCOPE] Reusing existing latest output: {latest_output}")
        return {
            'raw_output': raw_output,
            'latest_output': latest_output,
            'transcripts_csv': transcripts_csv,
            'cellpose_mask_path': mask_path,
        }

    if raw_output.exists() and (not FORCE_RERUN):
        status(f"[MERSCOPE] Found existing raw ProSeg output; converting to latest only")
        latest_out = convert_to_latest_zarr(raw_output, latest_output)
        return {
            'raw_output': raw_output,
            'latest_output': Path(latest_out),
            'transcripts_csv': transcripts_csv,
            'cellpose_mask_path': mask_path,
        }

    status('[MERSCOPE] Loading SpatialData (lazy zarr)')
    sdata = sd.read_zarr(MERSCOPE_ZARR_PATH)

    M = np.loadtxt(MERSCOPE_TRANSFORM_PATH)

    plane_keys = list_plane_keys(sdata.images, prefix=MERSCOPE_IMAGE_KEY_PREFIX)
    z0, z1 = MERSCOPE_Z_RANGE
    selected_keys = [k for z, k in plane_keys if z0 <= z <= z1]
    if not selected_keys:
        raise ValueError('[MERSCOPE] No image planes selected; check prefix and z-range.')

    status(f"[MERSCOPE] Selected {len(selected_keys)} z-planes from range {MERSCOPE_Z_RANGE}")
    plane_sources, height, width, use_channels = prepare_merscope_plane_sources(
        sdata,
        selected_keys=selected_keys,
        requested_channels=MERSCOPE_CHANNELS,
    )
    status(f"[MERSCOPE] Image shape={height}x{width}, channels={use_channels}")

    def _fetch_tile(y0, y1, x0, x1):
        return fetch_merscope_projected_tile(plane_sources, y0, y1, x0, x1)

    mask_path = run_tiled_cellpose_segmentation(
        fetch_tile_fn=_fetch_tile,
        height=height,
        width=width,
        dataset_name='MERSCOPE',
        output_mask_path=mask_path,
        cp_params=CELLPOSE_PARAMS,
        mask_filter_params=MASK_FILTER_PARAMS,
        tile_size_candidates=CELLPOSE_TILE_SIZE_CANDIDATES,
        overlap=CELLPOSE_TILE_OVERLAP,
        min_tile_size=CELLPOSE_MIN_TILE_SIZE,
        status_every_tiles=CELLPOSE_STATUS_EVERY_TILES,
        apply_filter_per_tile=bool(CELLPOSE_FILTER_PER_TILE),
    )

    scale_factor = 1.0
    cellpose_x_transform, cellpose_y_transform = build_cellpose_affine_to_microns(M, scale_factor, 0.0, 0.0)
    status(f"[MERSCOPE] mask->micron x_transform={cellpose_x_transform}")
    status(f"[MERSCOPE] mask->micron y_transform={cellpose_y_transform}")

    points_key = list(sdata.points.keys())[0]
    pts_obj = sdata.points[points_key]
    x_col = _resolve_col(pts_obj, ['x', 'global_x', 'x_location'])
    y_col = _resolve_col(pts_obj, ['y', 'global_y', 'y_location'])
    z_col = _resolve_col(pts_obj, ['z', 'global_z', 'z_location'], required=False)
    gene_col = _resolve_col(pts_obj, ['gene', 'feature_name', 'target'])

    mask_mmap = np.load(mask_path, mmap_mode='r')

    prep_stats = write_proseg_csv_from_points(
        points_obj=pts_obj,
        csv_path=transcripts_csv,
        masks=mask_mmap,
        x_transform=cellpose_x_transform,
        y_transform=cellpose_y_transform,
        x_col=x_col,
        y_col=y_col,
        z_col=z_col,
        gene_col=gene_col,
        qv_col=None,
        min_qv=None,
        chunk_rows=TRANSCRIPT_CHUNK_ROWS,
        dataset_name='MERSCOPE',
        status_every_chunks=TRANSCRIPT_STATUS_EVERY_CHUNKS,
    )

    status(
        f"[MERSCOPE] Seeded transcripts for ProSeg: "
        f"{prep_stats['n_seeded']:,} ({prep_stats['pct_seeded']:.2f}%)"
    )

    del mask_mmap, pts_obj, sdata, plane_sources
    force_release(note='after MERSCOPE preprocessing, before ProSeg')
    enforce_memory_limit(stage='before ProSeg MERSCOPE')

    proseg_params = dict(BASE_PROSEG_PARAMS)
    proseg_params.update(DATASET_PROSEG_OVERRIDES['MERSCOPE'])

    status('[MERSCOPE] Running ProSeg')
    raw_out = run_proseg_refinement(
        transcripts_df=transcripts_csv,
        output_path=raw_output,
        proseg_binary=PROSEG_BINARY,
        x_col='x_micron',
        y_col='y_micron',
        z_col='z_micron',
        gene_col='feature_name',
        cell_id_col='cell_id',
        samples=proseg_params['samples'],
        burnin_voxel_size=proseg_params['burnin_voxel_size'],
        voxel_size=proseg_params['voxel_size'],
        voxel_layers=proseg_params['voxel_layers'],
        nuclear_reassignment_prob=proseg_params['nuclear_reassignment_prob'],
        diffusion_probability=proseg_params['diffusion_probability'],
        cell_compactness=proseg_params['cell_compactness'],
        expand_initialized_cells=proseg_params['expand_initialized_cells'],
        use_cell_initialization=proseg_params['use_cell_initialization'],
        prior_seg_reassignment_prob=proseg_params['prior_seg_reassignment_prob'],
        max_transcript_nucleus_distance=proseg_params['max_transcript_nucleus_distance'],
        diffusion_sigma_far=proseg_params['diffusion_sigma_far'],
        cellpose_masks=mask_path,
        cellpose_x_transform=cellpose_x_transform,
        cellpose_y_transform=cellpose_y_transform,
        num_threads=proseg_params['num_threads'],
        overwrite=True,
        logger=None,
    )

    latest_out = convert_to_latest_zarr(raw_out, latest_output)
    status(f"[MERSCOPE] Wrote latest output: {latest_out}")

    force_release(note='after MERSCOPE full run')

    return {
        'raw_output': Path(raw_out),
        'latest_output': Path(latest_out),
        'transcripts_csv': transcripts_csv,
        'cellpose_mask_path': mask_path,
    }


In [ ]:
def run_full_xenium(output_dir):
    _ensure_no_rescale()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    raw_output = output_dir / 'proseg_base_raw.zarr'
    latest_output = output_dir / 'proseg_base_latest.zarr'
    transcripts_csv = output_dir / 'transcripts_for_proseg.csv'
    mask_path = output_dir / 'cellpose_masks_tiled.npy'

    if latest_output.exists() and (not FORCE_RERUN):
        status(f"[XENIUM] Reusing existing latest output: {latest_output}")
        return {
            'raw_output': raw_output,
            'latest_output': latest_output,
            'transcripts_csv': transcripts_csv,
            'cellpose_mask_path': mask_path,
        }

    if raw_output.exists() and (not FORCE_RERUN):
        status(f"[XENIUM] Found existing raw ProSeg output; converting to latest only")
        latest_out = convert_to_latest_zarr(raw_output, latest_output)
        return {
            'raw_output': raw_output,
            'latest_output': Path(latest_out),
            'transcripts_csv': transcripts_csv,
            'cellpose_mask_path': mask_path,
        }

    if not XENIUM_DIR.exists():
        raise FileNotFoundError(f'[XENIUM] directory not found: {XENIUM_DIR}')

    spec = json.loads(XENIUM_SPEC_PATH.read_text())
    microns_per_pixel = float(spec['pixel_size'])
    M = np.array([
        [1.0 / microns_per_pixel, 0.0, 0.0],
        [0.0, 1.0 / microns_per_pixel, 0.0],
        [0.0, 0.0, 1.0],
    ], dtype=float)

    need_tiled_cellpose = FORCE_RERUN or (not mask_path.exists())

    status('[XENIUM] Loading SpatialData from raw Xenium folder (lazy)')
    sdata = xenium_reader(
        XENIUM_DIR,
        cells_table=False,
        cells_as_circles=False,
        cells_boundaries=False,
        nucleus_boundaries=False,
        cells_labels=False,
        nucleus_labels=False,
        transcripts=True,
        morphology_focus=bool(need_tiled_cellpose),
        morphology_mip=False,
        aligned_images=False,
    )

    if need_tiled_cellpose:
        img_key = 'morphology_focus' if 'morphology_focus' in sdata.images else list(sdata.images.keys())[0]
        img_source = build_image_source(
            sdata.images[img_key],
            requested_channels=XENIUM_CHANNELS,
            as_float32=True,
        )
        height, width, _ = img_source['shape']
        status(f"[XENIUM] Image shape={height}x{width}, channels used={img_source['channels']}")

        def _fetch_tile(y0, y1, x0, x1):
            return fetch_tile(img_source, y0, y1, x0, x1)

        mask_path = run_tiled_cellpose_segmentation(
            fetch_tile_fn=_fetch_tile,
            height=height,
            width=width,
            dataset_name='XENIUM',
            output_mask_path=mask_path,
            cp_params=CELLPOSE_PARAMS,
            mask_filter_params=MASK_FILTER_PARAMS,
            tile_size_candidates=CELLPOSE_TILE_SIZE_CANDIDATES,
            overlap=CELLPOSE_TILE_OVERLAP,
            min_tile_size=CELLPOSE_MIN_TILE_SIZE,
            status_every_tiles=CELLPOSE_STATUS_EVERY_TILES,
            apply_filter_per_tile=bool(CELLPOSE_FILTER_PER_TILE),
        )

        del img_source
        force_release(note='after XENIUM tiled Cellpose image stage')
    else:
        status(f"[XENIUM] Reusing existing tiled Cellpose mask: {mask_path}")
        mask_mmap_check = np.load(mask_path, mmap_mode='r')
        status(f"[XENIUM] Existing mask shape={mask_mmap_check.shape}, dtype={mask_mmap_check.dtype}")
        del mask_mmap_check

    scale_factor = 1.0
    cellpose_x_transform, cellpose_y_transform = build_cellpose_affine_to_microns(M, scale_factor, 0.0, 0.0)
    status(f"[XENIUM] mask->micron x_transform={cellpose_x_transform}")
    status(f"[XENIUM] mask->micron y_transform={cellpose_y_transform}")

    points_key = list(sdata.points.keys())[0]
    pts_obj = sdata.points[points_key]

    x_col = _resolve_col(pts_obj, ['x', 'x_location'])
    y_col = _resolve_col(pts_obj, ['y', 'y_location'])
    z_col = _resolve_col(pts_obj, ['z', 'z_location'], required=False)
    gene_col = _resolve_col(pts_obj, ['feature_name', 'gene', 'target'])
    qv_col = first_existing_col(pts_obj, ['qv', 'quality', 'quality_value'])

    mask_mmap = np.load(mask_path, mmap_mode='r')

    prep_stats = write_proseg_csv_from_points(
        points_obj=pts_obj,
        csv_path=transcripts_csv,
        masks=mask_mmap,
        x_transform=cellpose_x_transform,
        y_transform=cellpose_y_transform,
        x_col=x_col,
        y_col=y_col,
        z_col=z_col,
        gene_col=gene_col,
        qv_col=qv_col,
        min_qv=XENIUM_MIN_QV,
        chunk_rows=TRANSCRIPT_CHUNK_ROWS,
        dataset_name='XENIUM',
        status_every_chunks=TRANSCRIPT_STATUS_EVERY_CHUNKS,
    )

    status(
        f"[XENIUM] Seeded transcripts for ProSeg: "
        f"{prep_stats['n_seeded']:,} ({prep_stats['pct_seeded']:.2f}%)"
    )

    del mask_mmap, pts_obj, sdata
    force_release(note='after XENIUM preprocessing, before ProSeg')
    enforce_memory_limit(stage='before ProSeg XENIUM')

    proseg_params = dict(BASE_PROSEG_PARAMS)
    proseg_params.update(DATASET_PROSEG_OVERRIDES['XENIUM'])

    status('[XENIUM] Running ProSeg')
    raw_out = run_proseg_refinement(
        transcripts_df=transcripts_csv,
        output_path=raw_output,
        proseg_binary=PROSEG_BINARY,
        x_col='x_micron',
        y_col='y_micron',
        z_col='z_micron',
        gene_col='feature_name',
        cell_id_col='cell_id',
        samples=proseg_params['samples'],
        burnin_voxel_size=proseg_params['burnin_voxel_size'],
        voxel_size=proseg_params['voxel_size'],
        voxel_layers=proseg_params['voxel_layers'],
        nuclear_reassignment_prob=proseg_params['nuclear_reassignment_prob'],
        diffusion_probability=proseg_params['diffusion_probability'],
        cell_compactness=proseg_params['cell_compactness'],
        expand_initialized_cells=proseg_params['expand_initialized_cells'],
        use_cell_initialization=proseg_params['use_cell_initialization'],
        prior_seg_reassignment_prob=proseg_params['prior_seg_reassignment_prob'],
        max_transcript_nucleus_distance=proseg_params['max_transcript_nucleus_distance'],
        diffusion_sigma_far=proseg_params['diffusion_sigma_far'],
        cellpose_masks=mask_path,
        cellpose_x_transform=cellpose_x_transform,
        cellpose_y_transform=cellpose_y_transform,
        num_threads=proseg_params['num_threads'],
        overwrite=True,
        logger=None,
    )

    latest_out = convert_to_latest_zarr(raw_out, latest_output)
    status(f"[XENIUM] Wrote latest output: {latest_out}")

    force_release(note='after XENIUM full run')

    return {
        'raw_output': Path(raw_out),
        'latest_output': Path(latest_out),
        'transcripts_csv': transcripts_csv,
        'cellpose_mask_path': mask_path,
    }


In [ ]:
# -------------------------------
# Execute full runs
# -------------------------------
run_outputs = {}
FORCE_RERUN = False

if RUN_MERSCOPE:
    status('[PIPELINE] Starting full MERSCOPE run')
    t0 = time.time()
    run_outputs['MERSCOPE'] = run_full_merscope(OUTPUT_ROOT / 'merscope_full_base_proseg')
    status(f"[PIPELINE] MERSCOPE done in {(time.time()-t0)/60:.1f} min")

    # Explicit RAM cleanup before Xenium.
    force_release(note='after MERSCOPE dataset completed')
    enforce_memory_limit(stage='post-MERSCOPE cleanup')

if RUN_XENIUM:
    status('[PIPELINE] Starting full XENIUM run')
    t0 = time.time()
    run_outputs['XENIUM'] = run_full_xenium(OUTPUT_ROOT / 'xenium_full_base_proseg')
    status(f"[PIPELINE] XENIUM done in {(time.time()-t0)/60:.1f} min")

    force_release(note='after XENIUM dataset completed')
    enforce_memory_limit(stage='post-XENIUM cleanup')

print('\nRun outputs:')
for k, v in run_outputs.items():
    print('-', k, v)


In [ ]:
# -------------------------------
# Enrich latest SpatialData outputs
# -------------------------------
import xarray as xr
from shapely.affinity import affine_transform
from shapely.geometry import Polygon
from reseg_tools import masks_to_polygons
from spatialdata.models import ShapesModel

ENRICH_FORCE_RERUN = FORCE_RERUN
ENRICH_KEEP_BACKUP = False
ENRICH_AFTER_RUN = True
ENRICH_CELLPOSE_POLYGON_N_JOBS = 16
ENRICH_CELLPOSE_POLYGON_SHOW_PROGRESS = True

MOSAIK_PROSEG_SHAPE_NAME = 'MOSAIK_proseg'
MOSAIK_CELLPOSE_SHAPE_NAME = 'MOSAIK_cellpose'
MERSCOPE_OLD_SHAPE_NAME = 'merscope_cell_boundaries'
XENIUM_OLD_CELL_SHAPE_NAME = 'xenium_cell_boundaries'
XENIUM_OLD_NUCLEUS_SHAPE_NAME = 'xenium_nucleus'
ORIGINAL_TABLE_NAME = 'table_original'
MERSCOPE_ZPROJ_IMAGE_NAME = 'MERSCOPE_z_projection'


def _get_latest_output_map():
    out = {
        'MERSCOPE': OUTPUT_ROOT / 'merscope_full_base_proseg' / 'proseg_base_latest.zarr',
        'XENIUM': OUTPUT_ROOT / 'xenium_full_base_proseg' / 'proseg_base_latest.zarr',
    }
    if isinstance(globals().get('run_outputs', None), dict):
        for ds, info in run_outputs.items():
            ds_key = str(ds).upper()
            if isinstance(info, dict) and ('latest_output' in info):
                cand = Path(info['latest_output'])
                if cand.exists():
                    out[ds_key] = cand
                else:
                    status(f"[{ds_key}] Ignoring stale run_outputs latest path (missing): {cand}")
    return out


def _prefer_existing_path(primary, fallback=None, dataset_name='DATASET', label='path'):
    p1 = Path(primary) if primary is not None else None
    p2 = Path(fallback) if fallback is not None else None

    if p1 is not None and p1.exists():
        return p1
    if p2 is not None and p2.exists():
        status(f"[{dataset_name}] Using fallback {label}: {p2}")
        return p2

    if p1 is not None and p2 is not None:
        status(f"[{dataset_name}] Neither primary nor fallback {label} exists: primary={p1} fallback={p2}")
    elif p1 is not None:
        status(f"[{dataset_name}] Missing {label}: {p1}")

    return p1


def _delete_if_exists(mapping, key):
    try:
        if key in mapping:
            del mapping[key]
    except Exception:
        pass


def _set_element(mapping, key, value, force=False):
    if (key in mapping) and (not force):
        return False
    if key in mapping and force:
        _delete_if_exists(mapping, key)
    mapping[key] = value
    return True


def _to_cyx(img_like):
    da = _get_image_dataarray(img_like)
    dims = tuple(str(d) for d in da.dims)

    if all(d in dims for d in ('c', 'y', 'x')):
        return da.transpose('c', 'y', 'x')
    if all(d in dims for d in ('y', 'x', 'c')):
        return da.transpose('c', 'y', 'x')
    if all(d in dims for d in ('y', 'x')):
        return da.expand_dims(c=['c0']).transpose('c', 'y', 'x')

    raise ValueError(f'Unsupported image dims for conversion to (c,y,x): {dims}')


def _find_shape_key(keys, required_terms):
    keys = [str(k) for k in keys]
    for k in keys:
        lk = k.lower()
        if all(t in lk for t in required_terms):
            return k
    return None


def _parse_shapes_with_template(gdf, template_shape=None):
    transformations = None
    if template_shape is not None and hasattr(template_shape, 'attrs'):
        transformations = template_shape.attrs.get('transform', None)

    if transformations is not None:
        try:
            return ShapesModel.parse(gdf, transformations=transformations)
        except TypeError:
            # spatialdata versions may differ in parse signature
            pass

    return ShapesModel.parse(gdf)


def _prepare_original_table(adata, target_region_name):
    tbl = adata.copy()

    spatial_attrs = dict(tbl.uns.get('spatialdata_attrs', {}))
    region_key = str(spatial_attrs.get('region_key', 'region'))
    instance_key = spatial_attrs.get('instance_key', None)

    if region_key not in tbl.obs.columns:
        region_key = 'region'
        tbl.obs[region_key] = target_region_name

    tbl.obs[region_key] = pd.Categorical([target_region_name] * tbl.n_obs)

    if (instance_key is None) or (instance_key not in tbl.obs.columns):
        for cand in ['cell_id', 'cell', 'cell_ID', 'region']:
            if cand in tbl.obs.columns:
                instance_key = cand
                break

    if (instance_key is None) or (instance_key not in tbl.obs.columns):
        instance_key = 'cell_id'
        tbl.obs[instance_key] = tbl.obs_names.astype(str)

    tbl.uns['spatialdata_attrs'] = {
        'region': target_region_name,
        'region_key': region_key,
        'instance_key': instance_key,
    }
    return tbl


def _cellpose_gdf_from_mask(mask_path, x_transform, y_transform, dataset_name):
    if not Path(mask_path).exists():
        raise FileNotFoundError(f'[{dataset_name}] Missing Cellpose mask file: {mask_path}')

    status(f"[{dataset_name}] Building {MOSAIK_CELLPOSE_SHAPE_NAME} from {mask_path}")
    mask_mmap = np.load(mask_path, mmap_mode='r')

    polygons_px = masks_to_polygons(
        mask_mmap,
        factor_rescale=0,
        n_jobs=int(ENRICH_CELLPOSE_POLYGON_N_JOBS),
        show_progress=bool(ENRICH_CELLPOSE_POLYGON_SHOW_PROGRESS),
    )

    coeffs = [
        float(x_transform[0]), float(x_transform[1]),
        float(y_transform[0]), float(y_transform[1]),
        float(x_transform[2]), float(y_transform[2]),
    ]

    polygons_um = []
    for poly in tqdm(polygons_px, desc=f'[{dataset_name}] cellpose polygons->microns', unit='poly'):
        if poly is None or poly.is_empty:
            continue
        p = affine_transform(poly, coeffs)
        if p is not None and (not p.is_empty):
            polygons_um.append(p)

    del mask_mmap, polygons_px
    force_release(note=f'after {dataset_name} cellpose polygon conversion')

    if len(polygons_um) == 0:
        raise RuntimeError(f'[{dataset_name}] No valid polygons were created from Cellpose masks.')

    gdf = gpd.GeoDataFrame(
        {
            'cell_id': [f'cellpose_{i+1}' for i in range(len(polygons_um))],
            'geometry': polygons_um,
        },
        geometry='geometry',
    )
    return gdf


def _dataset_cellpose_transform(dataset_name):
    if dataset_name == 'MERSCOPE':
        M = np.loadtxt(MERSCOPE_TRANSFORM_PATH)
        return build_cellpose_affine_to_microns(M, scale_factor=1.0, x0=0.0, y0=0.0)

    if dataset_name == 'XENIUM':
        spec = json.loads(XENIUM_SPEC_PATH.read_text())
        mpp = float(spec['pixel_size'])
        M = np.array(
            [[1.0 / mpp, 0.0, 0.0],
             [0.0, 1.0 / mpp, 0.0],
             [0.0, 0.0, 1.0]],
            dtype=float,
        )
        return build_cellpose_affine_to_microns(M, scale_factor=1.0, x0=0.0, y0=0.0)

    raise ValueError(f'Unknown dataset: {dataset_name}')


def _load_xenium_original_table_from_matrix(xenium_dir):
    matrix_dir = Path(xenium_dir) / 'cell_feature_matrix'
    mtx_path = matrix_dir / 'matrix.mtx.gz'
    features_path = matrix_dir / 'features.tsv.gz'
    barcodes_path = matrix_dir / 'barcodes.tsv.gz'

    if not (mtx_path.exists() and features_path.exists() and barcodes_path.exists()):
        status(
            f"[XENIUM] Fallback matrix files not found under {matrix_dir}; "
            "cannot build table_original from cell_feature_matrix."
        )
        return None

    try:
        import anndata as ad
        from scipy.io import mmread
        from scipy import sparse as sps
    except Exception as exc:
        status(f"[XENIUM] Could not import matrix-table dependencies (anndata/scipy): {exc}")
        return None

    status('[XENIUM] Loading fallback original table from cell_feature_matrix/* (mtx)')

    barcodes = pd.read_csv(barcodes_path, sep='	', header=None, compression='gzip').iloc[:, 0].astype(str).to_numpy()
    feats = pd.read_csv(features_path, sep='	', header=None, compression='gzip')
    feature_ids = feats.iloc[:, 0].astype(str).to_numpy()
    gene_names = feats.iloc[:, 1].astype(str).to_numpy() if feats.shape[1] > 1 else feature_ids.copy()

    X = mmread(str(mtx_path))
    if not sps.issparse(X):
        X = sps.csr_matrix(X)
    X = X.tocsr()

    # 10x matrix is usually genes x cells; transpose to cells x genes.
    if X.shape[1] == len(barcodes):
        X = X.T.tocsr()
    elif X.shape[0] != len(barcodes):
        status(
            f"[XENIUM] WARNING: barcode count ({len(barcodes)}) doesn't match matrix dimensions {tuple(X.shape)}; "
            "attempting best-effort alignment."
        )

    # Best-effort shape alignment
    if X.shape[0] != len(barcodes):
        n = min(X.shape[0], len(barcodes))
        X = X[:n, :]
        barcodes = barcodes[:n]

    if X.shape[1] != len(gene_names):
        if X.shape[0] == len(gene_names) and X.shape[1] == len(barcodes):
            X = X.T.tocsr()
        if X.shape[1] < len(gene_names):
            gene_names = gene_names[:X.shape[1]]
            feature_ids = feature_ids[:X.shape[1]]
        elif X.shape[1] > len(gene_names):
            gene_names = np.array([f'gene_{i}' for i in range(X.shape[1])], dtype=object)
            feature_ids = gene_names.copy()

    adata = ad.AnnData(X=X)
    adata.obs_names = pd.Index(barcodes.astype(str), name='cell_id')
    adata.var_names = pd.Index(pd.Series(gene_names).astype(str).to_numpy(), name='gene')
    adata.var_names_make_unique()
    adata.var['gene'] = adata.var_names.astype(str)

    if len(feature_ids) >= adata.n_vars:
        adata.var['feature_id'] = np.array(feature_ids[:adata.n_vars], dtype=object)
    if feats.shape[1] > 2 and len(feats) >= adata.n_vars:
        adata.var['feature_type'] = feats.iloc[:adata.n_vars, 2].astype(str).to_numpy()

    status(f"[XENIUM] Fallback table loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
    return adata


def _load_xenium_boundary_shapes_from_csv(xenium_dir, which='cell'):
    which = str(which).lower()
    if which.startswith('nuc'):
        csv_path = Path(xenium_dir) / 'nucleus_boundaries.csv.gz'
        desc = 'nucleus'
    else:
        csv_path = Path(xenium_dir) / 'cell_boundaries.csv.gz'
        desc = 'cell'

    if not csv_path.exists():
        status(f"[XENIUM] Missing {desc} boundary file: {csv_path}")
        return gpd.GeoDataFrame({'cell_id': [], 'geometry': []}, geometry='geometry')

    status(f"[XENIUM] Loading {desc} boundaries from {csv_path.name}")
    df = pd.read_csv(csv_path, compression='gzip', usecols=['cell_id', 'vertex_x', 'vertex_y', 'label_id'])

    df['label_id'] = pd.to_numeric(df['label_id'], errors='coerce')
    df['vertex_x'] = pd.to_numeric(df['vertex_x'], errors='coerce')
    df['vertex_y'] = pd.to_numeric(df['vertex_y'], errors='coerce')
    df = df.dropna(subset=['label_id', 'vertex_x', 'vertex_y'])

    polygons = []
    cell_ids = []

    grouped = df.groupby('label_id', sort=False)
    for label_id, grp in tqdm(grouped, total=grouped.ngroups, desc=f'[XENIUM] {desc} polygons', unit='poly'):
        xy = grp[['vertex_x', 'vertex_y']].to_numpy(dtype=float)
        if xy.shape[0] < 3:
            continue

        if not np.allclose(xy[0], xy[-1]):
            xy = np.vstack([xy, xy[0]])

        poly = Polygon(xy)
        if not poly.is_valid:
            poly = poly.buffer(0)
        if poly is None or poly.is_empty:
            continue

        polygons.append(poly)
        cell_ids.append(str(grp['cell_id'].iloc[0]))

    gdf = gpd.GeoDataFrame({'cell_id': cell_ids, 'geometry': polygons}, geometry='geometry')
    status(f"[XENIUM] {desc} boundaries loaded: {len(gdf):,} polygons")
    return gdf


def _load_original_source(dataset_name):
    if dataset_name == 'MERSCOPE':
        status('[MERSCOPE] Loading original source SpatialData for enrichment')
        return sd.read_zarr(MERSCOPE_ZARR_PATH)

    if dataset_name == 'XENIUM':
        status('[XENIUM] Loading original Xenium source for enrichment')
        return xenium_reader(
            XENIUM_DIR,
            cells_table=False,
            cells_as_circles=False,
            cells_boundaries=False,
            nucleus_boundaries=False,
            cells_labels=False,
            nucleus_labels=False,
            transcripts=False,
            morphology_focus=True,
            morphology_mip=False,
            aligned_images=False,
        )

    raise ValueError(f'Unknown dataset: {dataset_name}')


def _copy_merscope_images(dst_sdata, src_sdata, force=False):
    plane_pairs = list_plane_keys(src_sdata.images, prefix=MERSCOPE_IMAGE_KEY_PREFIX)
    plane_keys = [k for _, k in plane_pairs]
    if len(plane_keys) == 0:
        plane_keys = list(src_sdata.images.keys())

    copied = 0
    for k in plane_keys:
        copied += int(_set_element(dst_sdata.images, k, src_sdata.images[k], force=force))

    if len(plane_keys) > 0:
        status(f"[MERSCOPE] Building lazy z-projection from {len(plane_keys)} planes")
        proj = xr.concat([_to_cyx(src_sdata.images[k]) for k in plane_keys], dim='z').max(dim='z', keep_attrs=True)
        copied += int(_set_element(dst_sdata.images, MERSCOPE_ZPROJ_IMAGE_NAME, proj, force=force))

    return copied


def _copy_xenium_images(dst_sdata, src_sdata, force=False):
    copied = 0
    for k in list(src_sdata.images.keys()):
        copied += int(_set_element(dst_sdata.images, k, src_sdata.images[k], force=force))
    return copied


def _is_already_enriched(dst_sdata, dataset_name):
    req_shapes = {MOSAIK_PROSEG_SHAPE_NAME, MOSAIK_CELLPOSE_SHAPE_NAME}
    req_tables = {ORIGINAL_TABLE_NAME}

    if dataset_name == 'MERSCOPE':
        req_shapes.add(MERSCOPE_OLD_SHAPE_NAME)
        req_images = {MERSCOPE_ZPROJ_IMAGE_NAME}
    else:
        req_shapes.update({XENIUM_OLD_CELL_SHAPE_NAME, XENIUM_OLD_NUCLEUS_SHAPE_NAME})
        req_images = set()  # any copied Xenium image key is acceptable; we just don't require by name.

    has_shapes = req_shapes.issubset(set(map(str, dst_sdata.shapes.keys())))
    has_tables = req_tables.issubset(set(map(str, dst_sdata.tables.keys())))
    has_images = req_images.issubset(set(map(str, dst_sdata.images.keys())))

    return has_shapes and has_tables and has_images


def enrich_single_latest(dataset_name, latest_path, mask_path):
    dataset_name = str(dataset_name).upper()
    latest_path = Path(latest_path)
    mask_path = Path(mask_path)

    if not latest_path.exists():
        raise FileNotFoundError(f'[{dataset_name}] Latest zarr not found: {latest_path}')

    status(f"[{dataset_name}] Loading latest zarr for enrichment: {latest_path}")
    dst = sd.read_zarr(latest_path)

    if _is_already_enriched(dst, dataset_name) and (not ENRICH_FORCE_RERUN):
        status(f"[{dataset_name}] Already enriched; skipping (set ENRICH_FORCE_RERUN=True to rebuild)")
        del dst
        force_release(note=f'after enrichment skip {dataset_name}')
        return latest_path

    # 1) Add explicit MOSAIK_proseg boundary layer (copy from existing ProSeg shape)
    proseg_src_key = None
    for cand in ['cell_boundaries', 'cell_boundaries_refined', 'shapes', MOSAIK_PROSEG_SHAPE_NAME]:
        if cand in dst.shapes:
            proseg_src_key = cand
            break
    if proseg_src_key is None and len(dst.shapes) > 0:
        proseg_src_key = list(dst.shapes.keys())[0]

    if proseg_src_key is None:
        raise RuntimeError(f'[{dataset_name}] No shapes found in latest zarr to create {MOSAIK_PROSEG_SHAPE_NAME}')

    proseg_template = dst.shapes[proseg_src_key]
    _set_element(dst.shapes, MOSAIK_PROSEG_SHAPE_NAME, proseg_template.copy(), force=ENRICH_FORCE_RERUN)

    # 2) Add explicit MOSAIK_cellpose boundary layer from saved tiled mask
    x_tr, y_tr = _dataset_cellpose_transform(dataset_name)
    cp_gdf = _cellpose_gdf_from_mask(mask_path, x_tr, y_tr, dataset_name)
    cp_shapes = _parse_shapes_with_template(cp_gdf, template_shape=proseg_template)
    _set_element(dst.shapes, MOSAIK_CELLPOSE_SHAPE_NAME, cp_shapes, force=ENRICH_FORCE_RERUN)

    # 3) Load original source and copy old boundaries + old table + original images
    src = _load_original_source(dataset_name)

    if dataset_name == 'MERSCOPE':
        if len(src.shapes) == 0:
            raise RuntimeError('[MERSCOPE] No original shapes found to copy old boundaries.')
        old_key = list(src.shapes.keys())[0]
        _set_element(dst.shapes, MERSCOPE_OLD_SHAPE_NAME, src.shapes[old_key].copy(), force=ENRICH_FORCE_RERUN)

        img_added = _copy_merscope_images(dst, src, force=ENRICH_FORCE_RERUN)
        status(f"[MERSCOPE] Added/updated {img_added} image entries")

        if len(src.tables) > 0:
            old_tbl_key = list(src.tables.keys())[0]
            old_tbl = _prepare_original_table(src.tables[old_tbl_key], MERSCOPE_OLD_SHAPE_NAME)
            _set_element(dst.tables, ORIGINAL_TABLE_NAME, old_tbl, force=ENRICH_FORCE_RERUN)
        else:
            status('[MERSCOPE] WARNING: original source has no tables; table_original not added')

    elif dataset_name == 'XENIUM':
        # Boundaries: load directly from Xenium CSV exports to avoid spatialdata_io table/zarr bug.
        cell_gdf = _load_xenium_boundary_shapes_from_csv(XENIUM_DIR, which='cell')
        if len(cell_gdf) == 0:
            raise RuntimeError('[XENIUM] No cell boundaries could be parsed from cell_boundaries.csv.gz')
        cell_shapes = _parse_shapes_with_template(cell_gdf, template_shape=proseg_template)
        _set_element(dst.shapes, XENIUM_OLD_CELL_SHAPE_NAME, cell_shapes, force=ENRICH_FORCE_RERUN)

        nuc_gdf = _load_xenium_boundary_shapes_from_csv(XENIUM_DIR, which='nucleus')
        if len(nuc_gdf) > 0:
            nuc_shapes = _parse_shapes_with_template(nuc_gdf, template_shape=proseg_template)
            _set_element(dst.shapes, XENIUM_OLD_NUCLEUS_SHAPE_NAME, nuc_shapes, force=ENRICH_FORCE_RERUN)
        else:
            status('[XENIUM] WARNING: no nucleus boundaries parsed; xenium_nucleus not added')

        # Images: copy from xenium_reader output (images only).
        img_added = _copy_xenium_images(dst, src, force=ENRICH_FORCE_RERUN)
        status(f"[XENIUM] Added/updated {img_added} image entries")

        old_tbl_src = None
        if len(src.tables) > 0:
            old_tbl_key = list(src.tables.keys())[0]
            old_tbl_src = src.tables[old_tbl_key]
        else:
            status('[XENIUM] xenium_reader returned no table; trying fallback matrix loader')
            old_tbl_src = _load_xenium_original_table_from_matrix(XENIUM_DIR)

        if old_tbl_src is not None:
            old_tbl = _prepare_original_table(old_tbl_src, XENIUM_OLD_CELL_SHAPE_NAME)
            _set_element(dst.tables, ORIGINAL_TABLE_NAME, old_tbl, force=ENRICH_FORCE_RERUN)
        else:
            status('[XENIUM] WARNING: could not load original Xenium table; table_original not added')

    # 4) Write back (safe tmp write + replace)
    tmp_out = latest_path.parent / f"{latest_path.stem}__enrich_tmp.zarr"
    backup_out = latest_path.parent / f"{latest_path.stem}__pre_enrich_backup.zarr"

    if tmp_out.exists():
        shutil.rmtree(tmp_out)

    status(f"[{dataset_name}] Writing enriched zarr to temp path: {tmp_out}")
    dst.write(tmp_out, overwrite=True)

    del dst, src, cp_gdf
    force_release(note=f'after writing enriched temp zarr ({dataset_name})')

    if ENRICH_KEEP_BACKUP:
        if backup_out.exists():
            shutil.rmtree(backup_out)
        if latest_path.exists():
            shutil.move(str(latest_path), str(backup_out))
            status(f"[{dataset_name}] Backup saved: {backup_out}")
    else:
        if latest_path.exists():
            shutil.rmtree(latest_path)

    shutil.move(str(tmp_out), str(latest_path))
    status(f"[{dataset_name}] Enrichment complete: {latest_path}")
    return latest_path


if ENRICH_AFTER_RUN:
    latest_map = _get_latest_output_map()

    # Derive mask paths from run_outputs when available; fallback to default locations.
    mask_map = {
        'MERSCOPE': OUTPUT_ROOT / 'merscope_full_base_proseg' / 'cellpose_masks_tiled.npy',
        'XENIUM': OUTPUT_ROOT / 'xenium_full_base_proseg' / 'cellpose_masks_tiled.npy',
    }
    if isinstance(globals().get('run_outputs', None), dict):
        for ds, info in run_outputs.items():
            ds_key = str(ds).upper()
            if isinstance(info, dict) and ('cellpose_mask_path' in info):
                cand = Path(info['cellpose_mask_path'])
                if cand.exists():
                    mask_map[ds_key] = cand
                else:
                    status(f"[{ds_key}] Ignoring stale run_outputs mask path (missing): {cand}")

    enriched_outputs = {}
    for ds in ['MERSCOPE', 'XENIUM']:
        latest_default = latest_map.get(ds)
        latest = _prefer_existing_path(
            latest_default,
            fallback=(OUTPUT_ROOT / f"{ds.lower()}_full_base_proseg" / 'proseg_base_latest.zarr'),
            dataset_name=ds,
            label='latest output',
        )
        if latest is None or (not Path(latest).exists()):
            status(f"[{ds}] Skipping enrichment: latest output not found")
            continue

        mask_default = mask_map.get(ds)
        mask_path = _prefer_existing_path(
            mask_default,
            fallback=(OUTPUT_ROOT / f"{ds.lower()}_full_base_proseg" / 'cellpose_masks_tiled.npy'),
            dataset_name=ds,
            label='cellpose mask',
        )
        if mask_path is None or (not Path(mask_path).exists()):
            status(f"[{ds}] Skipping enrichment: cellpose mask not found")
            continue

        status(f"[{ds}] Enrichment inputs: latest={latest} | mask={mask_path}")
        t0 = time.time()
        enriched_outputs[ds] = enrich_single_latest(ds, latest, mask_path)
        status(f"[{ds}] Enrichment runtime: {(time.time() - t0)/60:.1f} min")

    print('Enriched outputs:')
    for ds, p in enriched_outputs.items():
        print('-', ds, p)

In [ ]:
# -------------------------------
# Build per-shape transcript assignments/tables
# -------------------------------
import anndata as ad
from scipy import sparse
from spatialdata.models import TableModel, ShapesModel

RUN_PER_SHAPE_ASSIGNMENT = True
ASSIGN_FORCE_RERUN = False
ASSIGN_CHUNK_ROWS = 750_000
ASSIGN_STATUS_EVERY_CHUNKS = 5
ASSIGN_TABLE_PREFIX = 'table_'
ASSIGN_SUMMARY_CSV = OUTPUT_ROOT / 'per_shape_assignment_summary.csv'


def _sanitize_table_key(shape_key):
    safe = re.sub(r'[^0-9a-zA-Z_]+', '_', str(shape_key)).strip('_')
    return f"{ASSIGN_TABLE_PREFIX}{safe}"


def _get_latest_output_map_for_assignment():
    out = {
        'MERSCOPE': OUTPUT_ROOT / 'merscope_full_base_proseg' / 'proseg_base_latest.zarr',
        'XENIUM': OUTPUT_ROOT / 'xenium_full_base_proseg' / 'proseg_base_latest.zarr',
    }
    if isinstance(globals().get('run_outputs', None), dict):
        for ds, info in run_outputs.items():
            ds_key = str(ds).upper()
            if isinstance(info, dict) and ('latest_output' in info):
                cand = Path(info['latest_output'])
                if cand.exists():
                    out[ds_key] = cand
    return out


def _resolve_points_cols(points_obj):
    x_col = first_existing_col(points_obj, ['x', 'x_micron', 'x_location', 'global_x', 'x_global_px', 'observed_x'])
    y_col = first_existing_col(points_obj, ['y', 'y_micron', 'y_location', 'global_y', 'y_global_px', 'observed_y'])
    gene_col = first_existing_col(points_obj, ['gene', 'feature_name', 'target'])
    qv_col = first_existing_col(points_obj, ['qv', 'quality', 'quality_value'])

    if x_col is None or y_col is None or gene_col is None:
        raise KeyError(
            f'Could not resolve points columns. x={x_col}, y={y_col}, gene={gene_col}. '
            f'Available: {list(points_obj.columns)}'
        )
    return x_col, y_col, gene_col, qv_col


def _parse_shapes_with_template_local(gdf, template_shape=None):
    transformations = None
    if template_shape is not None and hasattr(template_shape, 'attrs'):
        transformations = template_shape.attrs.get('transform', None)

    # Avoid double-specifying transforms: if gdf already carries them,
    # do not pass transformations=... to ShapesModel.parse.
    gdf_clean = gdf.copy()
    try:
        attrs = dict(getattr(gdf_clean, 'attrs', {}))
    except Exception:
        attrs = {}

    has_transform_in_gdf = ('transform' in attrs) and (attrs.get('transform') is not None)

    if transformations is not None and (not has_transform_in_gdf):
        try:
            return ShapesModel.parse(gdf_clean, transformations=transformations)
        except TypeError:
            # spatialdata versions may differ in parse signature
            pass
        except ValueError as exc:
            # Fallback when parser reports duplicated transformations
            if 'Transformations are both specified' in str(exc):
                gdf_no_t = gdf_clean.copy()
                try:
                    attrs2 = dict(getattr(gdf_no_t, 'attrs', {}))
                    attrs2.pop('transform', None)
                    gdf_no_t.attrs = attrs2
                except Exception:
                    pass
                return ShapesModel.parse(gdf_no_t, transformations=transformations)
            raise

    return ShapesModel.parse(gdf_clean)


def _ensure_shape_has_cell_id(sdata_obj, shape_key):
    """Return a normalized shape GeoDataFrame for assignment without mutating on-disk shapes."""
    shp = sdata_obj.shapes[shape_key]
    candidate = first_existing_col(shp, ['cell_id', 'cell', 'cells', 'cell_ID', 'region', 'label_id'])

    gdf = shp.copy()
    if candidate is None:
        gdf['cell_id'] = gdf.index.astype(str)
    else:
        gdf['cell_id'] = gdf[candidate].astype(str)

    gdf['cell_id'] = gdf['cell_id'].fillna('').astype(str)
    empty_mask = gdf['cell_id'].str.len() == 0
    if empty_mask.any():
        gdf.loc[empty_mask, 'cell_id'] = gdf.index.astype(str)[empty_mask]

    if gdf['cell_id'].duplicated().any():
        dup_rank = gdf.groupby('cell_id').cumcount()
        dup_mask = dup_rank > 0
        gdf.loc[dup_mask, 'cell_id'] = gdf.loc[dup_mask, 'cell_id'] + '__' + dup_rank[dup_mask].astype(str)
        status(f"[{shape_key}] Duplicate shape IDs detected; using suffixed IDs for assignment table")

    return gdf, 'cell_id'


def _build_gene_list_from_base_table(sdata_obj):
    if 'table' not in sdata_obj.tables:
        raise KeyError("Expected sdata.tables['table'] to exist for gene vocabulary")

    base_tbl = sdata_obj.tables['table']
    if 'gene' in base_tbl.var.columns:
        genes = base_tbl.var['gene'].astype(str)
    else:
        genes = base_tbl.var_names.astype(str)

    genes = pd.Index(pd.Series(genes).dropna().astype(str).unique())
    genes = genes.sort_values()
    return genes.tolist()


def _clone_table_for_region(table_obj, region_name):
    t = table_obj.copy()
    attrs = dict(t.uns.get('spatialdata_attrs', {}))
    region_key = str(attrs.get('region_key', 'region'))
    instance_key = attrs.get('instance_key', None)

    if region_key not in t.obs.columns:
        region_key = 'region'
        t.obs[region_key] = region_name

    t.obs[region_key] = pd.Categorical([region_name] * t.n_obs)

    if (instance_key is None) or (instance_key not in t.obs.columns):
        for cand in ['cell_id', 'cell', 'cells', 'cell_ID']:
            if cand in t.obs.columns:
                instance_key = cand
                break

    if (instance_key is None) or (instance_key not in t.obs.columns):
        instance_key = 'cell_id'
        t.obs[instance_key] = t.obs_names.astype(str)

    return TableModel.parse(
        t,
        region=region_name,
        region_key=region_key,
        instance_key=instance_key,
    )


def _compute_table_from_points_for_shape(
    dataset_name,
    points_obj,
    shape_gdf,
    shape_id_col,
    shape_key,
    gene_list,
):
    x_col, y_col, gene_col, _ = _resolve_points_cols(points_obj)

    gdf_shapes = shape_gdf[[shape_id_col, 'geometry']].copy()
    gdf_shapes = gdf_shapes[gdf_shapes.geometry.notna() & ~gdf_shapes.geometry.is_empty].copy()
    gdf_shapes[shape_id_col] = gdf_shapes[shape_id_col].astype(str)

    cell_ids = gdf_shapes[shape_id_col].astype(str).tolist()
    cell_to_idx = {cid: i for i, cid in enumerate(cell_ids)}

    genes = [str(g) for g in gene_list]
    gene_to_idx = {g: i for i, g in enumerate(genes)}

    _ = gdf_shapes.sindex  # build spatial index once

    counts_csr = sparse.csr_matrix((len(cell_ids), len(genes)), dtype=np.int64)

    n_input = 0
    n_used = 0
    n_assigned = 0

    chunk_iter = _iter_points_chunks(
        points_obj,
        columns=[x_col, y_col, gene_col],
        chunk_rows=ASSIGN_CHUNK_ROWS,
        desc=f'[{dataset_name}:{shape_key}] assign chunks',
    )

    for i, chunk in enumerate(chunk_iter, start=1):
        n_input += len(chunk)

        xv = pd.to_numeric(chunk[x_col], errors='coerce').to_numpy(np.float64)
        yv = pd.to_numeric(chunk[y_col], errors='coerce').to_numpy(np.float64)
        gv = chunk[gene_col].astype(str).to_numpy(dtype=object)

        valid = np.isfinite(xv) & np.isfinite(yv)
        valid &= pd.notna(gv)
        valid &= (gv != '')

        if np.any(valid):
            xvv = xv[valid]
            yvv = yv[valid]
            gvv = gv[valid]

            pts = gpd.GeoDataFrame(
                {'gene': pd.Series(gvv, dtype=str)},
                geometry=gpd.points_from_xy(xvv, yvv),
                crs=gdf_shapes.crs,
            )

            joined = gpd.sjoin(
                pts,
                gdf_shapes[[shape_id_col, 'geometry']],
                how='left',
                predicate='within',
            )

            cser = joined[shape_id_col].astype(str)
            assigned_mask = joined[shape_id_col].notna() & (cser != '') & (cser != '0')

            if assigned_mask.any():
                assigned_cells = cser.loc[assigned_mask].to_numpy(dtype=object)
                assigned_genes = joined.loc[assigned_mask, 'gene'].astype(str).to_numpy(dtype=object)

                cidx = np.fromiter((cell_to_idx.get(c, -1) for c in assigned_cells), dtype=np.int64, count=len(assigned_cells))
                gidx = np.fromiter((gene_to_idx.get(g, -1) for g in assigned_genes), dtype=np.int64, count=len(assigned_genes))
                keep = (cidx >= 0) & (gidx >= 0)

                if np.any(keep):
                    data = np.ones(int(np.sum(keep)), dtype=np.int64)
                    chunk_mat = sparse.coo_matrix(
                        (data, (cidx[keep], gidx[keep])),
                        shape=(len(cell_ids), len(genes)),
                    ).tocsr()
                    counts_csr = counts_csr + chunk_mat
                    n_assigned += int(np.sum(keep))

            n_used += len(xvv)
            del pts, joined, xvv, yvv, gvv

        if i % int(ASSIGN_STATUS_EVERY_CHUNKS) == 0:
            pct = 100.0 * n_assigned / max(n_used, 1)
            status(
                f"[{dataset_name}:{shape_key}] chunk={i} input={n_input:,} used={n_used:,} "
                f"assigned={n_assigned:,} ({pct:.2f}%)"
            )

        if i % int(MEMORY_CHECK_EVERY_CHUNKS) == 0:
            enforce_memory_limit(stage=f'{dataset_name}:{shape_key} chunk {i}')
            force_release()

        del chunk, xv, yv, gv, valid

    obs = pd.DataFrame(index=pd.Index(cell_ids, dtype=str, name='cell_id'))
    obs['cell_id'] = obs.index.astype(str)
    obs['region'] = pd.Categorical([shape_key] * len(obs), categories=[shape_key])

    var = pd.DataFrame(index=pd.Index(genes, dtype=str, name='gene'))
    var['gene'] = var.index.astype(str)

    adata = ad.AnnData(X=counts_csr, obs=obs, var=var)
    table = TableModel.parse(
        adata,
        region=shape_key,
        region_key='region',
        instance_key='cell_id',
    )

    summary = {
        'dataset': dataset_name,
        'shape_key': shape_key,
        'n_cells': int(len(cell_ids)),
        'n_genes': int(len(genes)),
        'n_points_input': int(n_input),
        'n_points_used': int(n_used),
        'n_points_assigned': int(n_assigned),
        'pct_assigned': float(100.0 * n_assigned / max(n_used, 1)),
    }
    return table, summary


def _shape_to_existing_table_source(shape_key):
    if shape_key in {'MOSAIK_proseg', 'cell_boundaries'}:
        return 'table'
    if shape_key in {'merscope_cell_boundaries', 'xenium_cell_boundaries'}:
        return 'table_original'
    return None


def run_per_shape_assignment_for_dataset(dataset_name, latest_path):
    dataset_name = str(dataset_name).upper()
    latest_path = Path(latest_path)

    status(f"[{dataset_name}] Loading enriched zarr for per-shape assignment: {latest_path}")
    sdata_obj = sd.read_zarr(latest_path)

    if len(sdata_obj.points) == 0:
        raise RuntimeError(f"[{dataset_name}] No points found in {latest_path}")

    points_key = list(sdata_obj.points.keys())[0]
    points_obj = sdata_obj.points[points_key]

    gene_list = _build_gene_list_from_base_table(sdata_obj)
    shape_keys = list(sdata_obj.shapes.keys())

    status(f"[{dataset_name}] Points key='{points_key}', shape layers={shape_keys}")

    summaries = []
    wrote_any = False

    for shape_key in tqdm(shape_keys, desc=f'[{dataset_name}] shape tables', unit='shape'):
        table_key = _sanitize_table_key(shape_key)

        table_exists = table_key in sdata_obj.tables
        if table_exists and (not ASSIGN_FORCE_RERUN):
            status(f"[{dataset_name}] Skipping existing table '{table_key}'")
            continue

        if table_exists and ASSIGN_FORCE_RERUN:
            try:
                sdata_obj.delete_element_from_disk(table_key)
            except Exception as exc:
                status(f"[{dataset_name}] delete_element_from_disk('{table_key}') warning: {exc}")
            try:
                del sdata_obj.tables[table_key]
            except Exception:
                pass

        shp, id_col = _ensure_shape_has_cell_id(sdata_obj, shape_key)

        source_table_key = _shape_to_existing_table_source(shape_key)
        if source_table_key in sdata_obj.tables:
            try:
                table_obj = _clone_table_for_region(sdata_obj.tables[source_table_key], shape_key)
                sdata_obj.tables[table_key] = table_obj
                sdata_obj.write_element(table_key, overwrite=False)
                wrote_any = True
                summaries.append({
                    'dataset': dataset_name,
                    'shape_key': shape_key,
                    'table_key': table_key,
                    'mode': f'copied_from_{source_table_key}',
                    'n_cells': int(sdata_obj.tables[table_key].n_obs),
                    'n_genes': int(sdata_obj.tables[table_key].n_vars),
                    'n_points_input': np.nan,
                    'n_points_used': np.nan,
                    'n_points_assigned': np.nan,
                    'pct_assigned': np.nan,
                })
                status(f"[{dataset_name}] Added '{table_key}' by cloning '{source_table_key}' (in-place write)")
                continue
            except Exception as exc:
                status(
                    f"[{dataset_name}] Clone from '{source_table_key}' failed for shape '{shape_key}' ({exc}); "
                    "falling back to point-in-polygon assignment."
                )

        table_obj, summary = _compute_table_from_points_for_shape(
            dataset_name=dataset_name,
            points_obj=points_obj,
            shape_gdf=shp,
            shape_id_col=id_col,
            shape_key=shape_key,
            gene_list=gene_list,
        )
        sdata_obj.tables[table_key] = table_obj
        sdata_obj.write_element(table_key, overwrite=False)
        wrote_any = True

        summary['table_key'] = table_key
        summary['mode'] = 'computed_sjoin'
        summaries.append(summary)
        status(
            f"[{dataset_name}] Added '{table_key}' | cells={summary['n_cells']:,} genes={summary['n_genes']:,} "
            f"assigned={summary['n_points_assigned']:,} ({summary['pct_assigned']:.2f}%) (in-place write)"
        )

    del sdata_obj
    if wrote_any:
        force_release(note=f'after per-shape in-place table writes ({dataset_name})')
        status(f"[{dataset_name}] Per-shape assignment tables written in-place to: {latest_path}")
    else:
        force_release(note=f'after per-shape assignment no-op ({dataset_name})')
        status(f"[{dataset_name}] No per-shape assignment updates were needed")

    return summaries


if RUN_PER_SHAPE_ASSIGNMENT:
    latest_map = _get_latest_output_map_for_assignment()
    all_summaries = []

    for ds in ['MERSCOPE', 'XENIUM']:
        latest = latest_map.get(ds)
        if latest is None or (not Path(latest).exists()):
            status(f"[{ds}] Skipping per-shape assignment: latest zarr missing")
            continue

        t0 = time.time()
        summaries = run_per_shape_assignment_for_dataset(ds, latest)
        all_summaries.extend(summaries)
        status(f"[{ds}] Per-shape assignment runtime: {(time.time()-t0)/60:.1f} min")

    if len(all_summaries) > 0:
        assign_summary_df = pd.DataFrame(all_summaries)
        display(assign_summary_df)
        assign_summary_df.to_csv(ASSIGN_SUMMARY_CSV, index=False)
        print('Saved assignment summary:', ASSIGN_SUMMARY_CSV)
    else:
        print('No per-shape assignment updates were made.')





In [ ]:
# -------------------------------
# Load outputs + compute QC tables
# -------------------------------
summary_csv = OUTPUT_ROOT / 'comparison_summary.csv'
geometry_pkl = OUTPUT_ROOT / 'comparison_geometry_metrics.pkl'
cell_pkl = OUTPUT_ROOT / 'comparison_cell_metrics.pkl'

qc_cache_exists = summary_csv.exists() and geometry_pkl.exists() and cell_pkl.exists()

if qc_cache_exists and (not FORCE_RERUN):
    status('[QC] Found cached QC tables on disk; loading (set FORCE_RERUN=True to recompute)')
    summary_df = pd.read_csv(summary_csv)
    geometry_df = pd.read_pickle(geometry_pkl)
    cell_df = pd.read_pickle(cell_pkl)
else:
    if len(run_outputs) == 0:
        raise ValueError(
            'No outputs found to recompute QC. '
            'Enable RUN_MERSCOPE and/or RUN_XENIUM, or disable FORCE_RERUN to use cached QC if present.'
        )

    qc_results = {}
    for dataset_name, out in tqdm(list(run_outputs.items()), desc='QC datasets', unit='dataset'):
        latest_path = out['latest_output']
        if not Path(latest_path).exists():
            raise FileNotFoundError(f'Missing latest output for {dataset_name}: {latest_path}')

        status(f"[QC] Computing metrics for {dataset_name}")
        qc_results[dataset_name] = compute_dataset_qc(latest_path, dataset_name)
        enforce_memory_limit(stage=f'QC {dataset_name}')

    summary_df = pd.DataFrame([qc_results[k]['summary'] for k in qc_results])
    geometry_df = pd.concat([qc_results[k]['geometry_metrics'] for k in qc_results], ignore_index=True)
    cell_df = pd.concat([qc_results[k]['cell_metrics'] for k in qc_results], ignore_index=True)

    summary_df.to_csv(summary_csv, index=False)
    geometry_df.to_pickle(geometry_pkl)
    cell_df.to_pickle(cell_pkl)
    print('Saved summary:', summary_csv)
    print('Saved geometry metrics:', geometry_pkl)
    print('Saved cell metrics:', cell_pkl)

display(summary_df)
print('geometry rows:', len(geometry_df), 'cell rows:', len(cell_df))

force_release(note='after QC table generation')



In [ ]:
# -------------------------------
# Geometry comparison plots
# -------------------------------
plot_dir = OUTPUT_ROOT / 'qc_plots'
plot_dir.mkdir(parents=True, exist_ok=True)

geom_metrics = [
    ('log10_area', 'log10(cell area)'),
    ('perimeter', 'perimeter'),
    ('eccentricity', 'eccentricity'),
    ('circularity', 'circularity'),
    ('solidity', 'solidity'),
    ('aspect_ratio', 'aspect ratio'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
axes = axes.ravel()
for ax, (col, title) in zip(axes, geom_metrics):
    d = geometry_df[[col, 'dataset']].replace([np.inf, -np.inf], np.nan).dropna()
    sns.histplot(
        data=d,
        x=col,
        hue='dataset',
        bins=80,
        stat='density',
        common_norm=False,
        element='step',
        fill=False,
        ax=ax,
    )
    ax.set_title(title)

geom_hist_path = plot_dir / 'geometry_hist_comparison.png'
plt.savefig(geom_hist_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', geom_hist_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
sns.violinplot(data=geometry_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['area']), x='dataset', y='area', cut=0, ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_title('Cell area (log scale)')

sns.violinplot(data=geometry_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['eccentricity']), x='dataset', y='eccentricity', cut=0, ax=axes[1])
axes[1].set_title('Eccentricity')

geom_violin_path = plot_dir / 'geometry_violin_comparison.png'
plt.savefig(geom_violin_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', geom_violin_path)

In [ ]:
# -------------------------------
# Transcript assignment / cell-level QC plots
# -------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

sns.violinplot(data=cell_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['transcripts_per_cell']),
               x='dataset', y='transcripts_per_cell', cut=0, ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_title('Assigned transcripts per cell (log scale)')

sns.violinplot(data=cell_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['genes_per_cell']),
               x='dataset', y='genes_per_cell', cut=0, ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Genes per cell (log scale)')

cell_violin_path = plot_dir / 'cell_violin_comparison.png'
plt.savefig(cell_violin_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', cell_violin_path)

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
bar = summary_df[['dataset', 'pct_assigned']].copy()
sns.barplot(data=bar, x='dataset', y='pct_assigned', ax=ax)
ax.set_ylabel('Assigned transcripts (%)')
ax.set_title('Transcript assignment rate')
for i, row in bar.reset_index(drop=True).iterrows():
    ax.text(i, row['pct_assigned'] + 0.5, f"{row['pct_assigned']:.2f}%", ha='center', va='bottom')

assign_bar_path = plot_dir / 'assignment_rate_bar.png'
plt.savefig(assign_bar_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', assign_bar_path)

display(summary_df)

In [ ]:
#xenium_sdata = sd.read_zarr(OUTPUT_ROOT / 'xenium_full_base_proseg' / 'proseg_base_latest.zarr')
#merscope_sdata = sd.read_zarr(OUTPUT_ROOT / 'merscope_full_base_proseg' / 'proseg_base_latest.zarr')


xenium_sdata = sd.read_zarr("/home/becalia/mnt/MOSAIK_analysis/tmp_full_reseg_compare_output/xenium_full_base_proseg/proseg_base_latest.zarr")
merscope_sdata = sd.read_zarr("/home/becalia/mnt/MOSAIK_analysis/tmp_full_reseg_compare_output/merscope_full_base_proseg/proseg_base_latest.zarr")

In [ ]:
merscope_sdata

In [ ]:
merscope_sdata.tables['table']

In [ ]:
merscope_sdata.tables['table_original']

In [ ]:
merscope_sdata.points['transcripts']

In [ ]:
xenium_sdata

In [ ]:
genes = xenium_sdata.tables['table'].var.gene

unique_genes = (
    genes.dropna()
    .astype(str)
    .loc[lambda s: ~s.str.startswith(("UnassignedCodeword", "NegControlCodeword", "NegControlProbe"))]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print(f"{len(unique_genes)} genes")
print(unique_genes)          # or: for g in unique_genes: print(g)


In [ ]:
# Gene-set comparison + per-gene scatter plots (un-normalized vs normalized)
from IPython.display import display
xenium_table = xenium_sdata.tables['table']
merscope_table = merscope_sdata.tables['table']


def _gene_totals_from_table(adata, gene_label='gene'):
    if gene_label in adata.var.columns:
        genes = adata.var[gene_label].astype(str)
    else:
        genes = adata.var_names.astype(str)

    totals = np.asarray(adata.X.sum(axis=0)).ravel()
    return pd.Series(totals, index=genes).groupby(level=0).sum()


def _gene_totals_from_points(sdata_obj, assigned_only=False):
    if len(sdata_obj.points) == 0:
        raise RuntimeError('No points found in spatialdata object.')

    points_key = list(sdata_obj.points.keys())[0]
    pts = sdata_obj.points[points_key]

    gene_col = first_existing_col(pts, ['gene', 'feature_name', 'target'])
    if gene_col is None:
        raise KeyError(f'Could not find gene column in points {points_key}: {list(pts.columns)}')

    assign_col = None
    if assigned_only:
        assign_col = first_existing_col(pts, ['assignment', 'cell', 'cell_id'])
        if assign_col is None:
            raise KeyError(f'Could not find assignment column in points {points_key}: {list(pts.columns)}')

    if hasattr(pts, 'npartitions') and hasattr(pts, 'partitions'):
        if assigned_only:
            work = pts[[gene_col, assign_col]]
            mask = work[assign_col].map_partitions(assignment_mask, meta=('assigned_mask', 'bool'))
            counts = work.loc[mask].groupby(gene_col).size().compute()
        else:
            counts = pts[[gene_col]].groupby(gene_col).size().compute()
    else:
        pdf = _to_pandas(pts)
        if assigned_only:
            mask = assignment_mask(pdf[assign_col])
            counts = pdf.loc[mask].groupby(gene_col).size()
        else:
            counts = pdf.groupby(gene_col).size()

    counts.index = counts.index.astype(str)
    return counts.groupby(level=0).sum().astype(float)


def _apply_dataset_filter(gene_counts, dataset_name):
    idx = gene_counts.index.astype(str)
    if dataset_name.upper() == 'XENIUM':
        keep = ~idx.str.contains('Blank', na=False)
    elif dataset_name.upper() == 'MERSCOPE':
        keep = ~idx.str.contains('UnassignedCodeword|NegControlCodeword|NegControlProbe', regex=True, na=False)
    else:
        keep = pd.Series(True, index=gene_counts.index)
    return gene_counts.loc[keep].copy()


def _normalize_counts(gene_counts):
    total = float(gene_counts.sum())
    if total <= 0:
        raise ValueError('Cannot normalize: total count is <= 0.')
    return gene_counts / total, total


def _compare_df(x_counts, m_counts):
    common = sorted(set(x_counts.index) & set(m_counts.index))
    return pd.DataFrame({
        'gene': common,
        'xenium': x_counts.reindex(common).values,
        'merscope': m_counts.reindex(common).values,
    })


def _fit_linear(x, y):
    if len(x) < 2:
        return np.nan, np.nan, np.nan
    m, b = np.polyfit(x, y, 1)
    yhat = m * x + b
    denom = np.sum((y - y.mean()) ** 2)
    r2 = np.nan if denom <= 0 else (1.0 - np.sum((y - yhat) ** 2) / denom)
    return float(m), float(b), float(r2)


def _plot_compare(ax, df, title, x_label, y_label, log_scale=True):
    if df.empty:
        ax.set_title(title + ' (no overlapping genes)')
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        return

    x = df['xenium'].to_numpy(dtype=float)
    y = df['merscope'].to_numpy(dtype=float)

    if log_scale:
        eps = 1e-12
        x_plot = np.clip(x, eps, None)
        y_plot = np.clip(y, eps, None)

        ax.scatter(x_plot, y_plot, s=22, alpha=0.75, edgecolor='none')
        ax.set_xscale('log')
        ax.set_yscale('log')

        lo = float(min(x_plot.min(), y_plot.min()))
        hi = float(max(x_plot.max(), y_plot.max()))
        ax.plot([lo, hi], [lo, hi], '--', linewidth=1.2, label='y = x')

        lx = np.log10(x_plot)
        ly = np.log10(y_plot)
        m, b, r2 = _fit_linear(lx, ly)
        if np.isfinite(m):
            x_line = np.logspace(np.log10(lo), np.log10(hi), 200)
            y_line = 10 ** (m * np.log10(x_line) + b)
            ax.plot(x_line, y_line, linewidth=1.5, label='best fit')
            txt = f"log10(y) = {m:.3f} * log10(x) + {b:.3f}\nR² = {r2:.3f}"
            ax.text(
                0.03, 0.97, txt,
                transform=ax.transAxes,
                va='top', ha='left',
                fontsize=9,
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.8)
            )

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
    else:
        ax.scatter(x, y, s=22, alpha=0.75, edgecolor='none')

        lo = 0.0
        hi = float(max(x.max(), y.max()))
        ax.plot([lo, hi], [lo, hi], '--', linewidth=1.2, label='y = x')

        m, b, r2 = _fit_linear(x, y)
        if np.isfinite(m):
            x_line = np.linspace(lo, hi, 200)
            y_line = m * x_line + b
            ax.plot(x_line, y_line, linewidth=1.5, label='best fit')
            txt = f"y = {m:.3f} * x + {b:.3e}\nR² = {r2:.3f}"
            ax.text(
                0.03, 0.97, txt,
                transform=ax.transAxes,
                va='top', ha='left',
                fontsize=9,
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.8)
            )

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)


status('[Gene Compare] Computing TOTAL per-gene transcript counts from points...')
x_total_all = _gene_totals_from_points(xenium_sdata, assigned_only=False)
m_total_all = _gene_totals_from_points(merscope_sdata, assigned_only=False)

status('[Gene Compare] Computing ASSIGNED per-gene transcript counts from tables...')
# Table counts represent transcripts assigned to cells
x_assigned_all = _gene_totals_from_table(xenium_table)
m_assigned_all = _gene_totals_from_table(merscope_table)

# Apply requested platform-specific gene exclusions
x_total = _apply_dataset_filter(x_total_all, 'XENIUM')
m_total = _apply_dataset_filter(m_total_all, 'MERSCOPE')
x_assigned = _apply_dataset_filter(x_assigned_all, 'XENIUM')
m_assigned = _apply_dataset_filter(m_assigned_all, 'MERSCOPE')

# 1:1 gene checks
xg_total, mg_total = set(x_total.index), set(m_total.index)
only_x_total = sorted(xg_total - mg_total)
only_m_total = sorted(mg_total - xg_total)

print(f"TOTAL genes after filter - Xenium: {len(xg_total)}, MERSCOPE: {len(mg_total)}, intersection: {len(xg_total & mg_total)}")
print('TOTAL 1:1 match:', 'YES' if (len(only_x_total) == 0 and len(only_m_total) == 0) else 'NO')
if len(only_x_total) > 0:
    print(f"  Only in Xenium total ({len(only_x_total)}): {only_x_total[:20]}{' ...' if len(only_x_total) > 20 else ''}")
if len(only_m_total) > 0:
    print(f"  Only in MERSCOPE total ({len(only_m_total)}): {only_m_total[:20]}{' ...' if len(only_m_total) > 20 else ''}")

xg_assigned, mg_assigned = set(x_assigned.index), set(m_assigned.index)
only_x_assigned = sorted(xg_assigned - mg_assigned)
only_m_assigned = sorted(mg_assigned - xg_assigned)

print(f"ASSIGNED genes after filter - Xenium: {len(xg_assigned)}, MERSCOPE: {len(mg_assigned)}, intersection: {len(xg_assigned & mg_assigned)}")
print('ASSIGNED 1:1 match:', 'YES' if (len(only_x_assigned) == 0 and len(only_m_assigned) == 0) else 'NO')

# Normalize by dataset total counts (within each filtered gene set)
x_total_norm, x_total_den = _normalize_counts(x_total)
m_total_norm, m_total_den = _normalize_counts(m_total)
x_assigned_norm, x_assigned_den = _normalize_counts(x_assigned)
m_assigned_norm, m_assigned_den = _normalize_counts(m_assigned)

print(f"Normalization denominators (TOTAL): Xenium={x_total_den:,.0f}, MERSCOPE={m_total_den:,.0f}")
print(f"Normalization denominators (ASSIGNED): Xenium={x_assigned_den:,.0f}, MERSCOPE={m_assigned_den:,.0f}")

# Build compare tables for un-normalized and normalized views
compare_total_raw = _compare_df(x_total, m_total)
compare_total_norm = _compare_df(x_total_norm, m_total_norm)
compare_assigned_raw = _compare_df(x_assigned, m_assigned)
compare_assigned_norm = _compare_df(x_assigned_norm, m_assigned_norm)

fig, axes = plt.subplots(2, 2, figsize=(15, 12), constrained_layout=True)
_plot_compare(
    axes[0, 0], compare_total_raw,
    'TOTAL per-gene counts (un-normalized, log)',
    x_label='Xenium total transcripts per gene',
    y_label='MERSCOPE total transcripts per gene',
    log_scale=True,
)
_plot_compare(
    axes[0, 1], compare_total_norm,
    'TOTAL per-gene counts (normalized, log)',
    x_label='Xenium normalized total per gene',
    y_label='MERSCOPE normalized total per gene',
    log_scale=True,
)
_plot_compare(
    axes[1, 0], compare_assigned_raw,
    'ASSIGNED per-gene counts (un-normalized, log)',
    x_label='Xenium assigned transcripts per gene',
    y_label='MERSCOPE assigned transcripts per gene',
    log_scale=True,
)
_plot_compare(
    axes[1, 1], compare_assigned_norm,
    'ASSIGNED per-gene counts (normalized, log)',
    x_label='Xenium normalized assigned per gene',
    y_label='MERSCOPE normalized assigned per gene',
    log_scale=True,
)
plt.show()

display(compare_total_raw.head())
display(compare_total_norm.head())
display(compare_assigned_raw.head())
display(compare_assigned_norm.head())


## Next steps

1. If runtime/memory is high, set `CELLPOSE_PARAMS['factor_rescale'] > 1` and rerun.
2. For faster iteration, set only one of `RUN_MERSCOPE` / `RUN_XENIUM` to `True`.
3. Tune `DATASET_PROSEG_OVERRIDES['XENIUM']['voxel_layers']` in the `2-8` range if needed.
4. Add additional QC metrics (e.g., nucleus distance distributions, per-gene assignment rates) if required.

In [ ]:
# Sanity plots: image background + all shape contours + ProSeg assigned/unassigned transcripts in 250 um crops
from shapely.geometry import box as shapely_box
from matplotlib.lines import Line2D
from pathlib import Path
import json

SANITY_CROP_SIZE_UM = 250.0
SANITY_MAX_TRANSCRIPTS_PER_PANEL = 2_000_000  # random subsample only if crop is very dense
SANITY_RANDOM_STATE = 42
SANITY_ASSIGNMENT_SHAPE_KEY = 'MOSAIK_proseg'  # set to a shapes key, or None to use points['assignment']


def _first_existing_col_local(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _to_pandas_local(df):
    if hasattr(df, 'compute'):
        return df.compute()
    return pd.DataFrame(df).copy()


def _assignment_mask_local(series):
    s = pd.Series(series)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float) > 0
    s = s.astype('string')
    bad = {'', '0', '-1', 'nan', 'None', '<NA>'}
    return s.notna() & ~s.isin(bad)


def _shape_geometry_only(shape_obj):
    if 'geometry' in shape_obj.columns:
        gdf = shape_obj[['geometry']].copy()
    else:
        gdf = gpd.GeoDataFrame({'geometry': shape_obj.geometry}, index=shape_obj.index)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    return gdf


def _reference_shape_key(sdata_obj):
    for k in ['MOSAIK_proseg', 'cell_boundaries', 'MOSAIK_cellpose']:
        if k in sdata_obj.shapes:
            return k
    if len(sdata_obj.shapes) == 0:
        raise RuntimeError('No shapes found in SpatialData object.')
    return list(sdata_obj.shapes.keys())[0]


def _bounded_interval(center, size, min_v, max_v):
    span = max_v - min_v
    if span <= size:
        return float(min_v), float(max_v)
    half = size / 2.0
    lo = center - half
    hi = center + half
    if lo < min_v:
        hi += (min_v - lo)
        lo = min_v
    if hi > max_v:
        lo -= (hi - max_v)
        hi = max_v
    return float(lo), float(hi)


def _choose_crop_bbox(sdata_obj, size_um=250.0, center_xy=None):
    ref_key = _reference_shape_key(sdata_obj)
    gdf = _shape_geometry_only(sdata_obj.shapes[ref_key])
    if len(gdf) == 0:
        raise RuntimeError(f'No non-empty geometries in shapes[{ref_key}]')

    minx, miny, maxx, maxy = gdf.total_bounds
    if center_xy is None:
        cx = 0.5 * (minx + maxx)
        cy = 0.5 * (miny + maxy)
    else:
        cx, cy = center_xy

    x0, x1 = _bounded_interval(cx, size_um, minx, maxx)
    y0, y1 = _bounded_interval(cy, size_um, miny, maxy)
    return (x0, y0, x1, y1), ref_key


def _crop_single_shape(sdata_obj, shape_key, bbox):
    gdf = _shape_geometry_only(sdata_obj.shapes[shape_key])
    crop_poly = shapely_box(*bbox)
    keep = gdf.geometry.intersects(crop_poly)
    return gdf.loc[keep].copy()


def _assign_points_by_shape(points_pdf, shapes_gdf):
    """Return one boolean assignment per transcript using point-in-polygon on a selected shape layer."""
    n = len(points_pdf)
    if n == 0:
        return np.zeros(0, dtype=bool)
    if len(shapes_gdf) == 0:
        return np.zeros(n, dtype=bool)

    # Track row ids explicitly so we can collapse any many-to-one sjoin duplicates.
    pts_gdf = gpd.GeoDataFrame(
        {'_row_id': np.arange(n, dtype=np.int64)},
        geometry=gpd.points_from_xy(points_pdf['x_um'].to_numpy(), points_pdf['y_um'].to_numpy()),
    )
    shp = shapes_gdf[['geometry']].copy()

    try:
        joined = gpd.sjoin(pts_gdf[['_row_id', 'geometry']], shp, how='left', predicate='within')
        matched = joined.loc[joined['index_right'].notna(), '_row_id'].to_numpy(dtype=np.int64, copy=False)
        out = np.zeros(n, dtype=bool)
        if matched.size:
            out[np.unique(matched)] = True
        return out
    except Exception:
        # Fallback for environments without a working vectorized spatial join backend.
        assigned = np.zeros(n, dtype=bool)
        try:
            sindex = shp.sindex
            use_sindex = sindex is not None
        except Exception:
            sindex = None
            use_sindex = False

        for i, pt in enumerate(pts_gdf.geometry.values):
            if use_sindex:
                cand = list(sindex.intersection(pt.bounds))
            else:
                cand = range(len(shp))
            if not cand:
                continue
            assigned[i] = any(shp.geometry.iloc[j].covers(pt) for j in cand)
        return assigned


def _crop_points(sdata_obj, bbox, max_points=None, random_state=42, assignment_shape_key=SANITY_ASSIGNMENT_SHAPE_KEY):
    if len(sdata_obj.points) == 0:
        raise RuntimeError('No points found in SpatialData object.')

    points_key = list(sdata_obj.points.keys())[0]
    pts = sdata_obj.points[points_key]

    x_col = _first_existing_col_local(pts, ['x', 'global_x', 'x_location'])
    y_col = _first_existing_col_local(pts, ['y', 'global_y', 'y_location'])
    assign_col = _first_existing_col_local(pts, ['assignment', 'cell', 'cell_id'])
    if x_col is None or y_col is None:
        raise KeyError(f'Could not resolve x/y columns in points[{points_key}]')

    x0, y0, x1, y1 = bbox
    cols = [x_col, y_col] + ([assign_col] if assign_col is not None else [])

    if hasattr(pts, 'npartitions') and hasattr(pts, 'partitions'):
        work = pts[cols]
        work = work[
            (work[x_col] >= x0) & (work[x_col] <= x1) &
            (work[y_col] >= y0) & (work[y_col] <= y1)
        ]
        pdf = work.compute()
    else:
        pdf = _to_pandas_local(pts[cols])
        pdf = pdf[
            (pdf[x_col] >= x0) & (pdf[x_col] <= x1) &
            (pdf[y_col] >= y0) & (pdf[y_col] <= y1)
        ].copy()

    pdf = pdf.rename(columns={x_col: 'x_um', y_col: 'y_um'})

    if assignment_shape_key is not None:
        if assignment_shape_key not in sdata_obj.shapes:
            raise KeyError(
                f"assignment_shape_key='{assignment_shape_key}' not found in shapes. "
                f"Available: {list(sdata_obj.shapes.keys())}"
            )
        shape_crop = _crop_single_shape(sdata_obj, assignment_shape_key, bbox)
        pdf['assigned'] = _assign_points_by_shape(pdf, shape_crop)
        assign_col = f'shape:{assignment_shape_key}'
    elif assign_col is not None and assign_col in pdf.columns:
        pdf['assigned'] = _assignment_mask_local(pdf[assign_col]).values
    else:
        pdf['assigned'] = True

    if max_points is not None and len(pdf) > max_points:
        pdf = pdf.sample(n=max_points, random_state=random_state)

    return pdf, points_key, assign_col


def _get_scale0_dataarray(image_elem):
    # SpatialData image elements are typically DataTree with scale pyramid.
    if hasattr(image_elem, 'keys') and 'scale0' in image_elem:
        node = image_elem['scale0']
        if hasattr(node, 'ds'):
            if 'image' in node.ds:
                return node.ds['image']
            if len(node.ds.data_vars) > 0:
                return next(iter(node.ds.data_vars.values()))
    if hasattr(image_elem, 'ds'):
        if 'image' in image_elem.ds:
            return image_elem.ds['image']
        if len(image_elem.ds.data_vars) > 0:
            return next(iter(image_elem.ds.data_vars.values()))
    return image_elem


def _pick_channel_name(channel_labels, preferred):
    lower = [str(c).lower() for c in channel_labels]
    for p in preferred:
        p_low = p.lower()
        for i, c in enumerate(lower):
            if c == p_low or p_low in c:
                return channel_labels[i]
    return None


def _norm01(arr):
    a = np.asarray(arr, dtype=np.float32)
    finite = np.isfinite(a)
    if not finite.any():
        return np.zeros_like(a, dtype=np.float32)
    lo, hi = np.percentile(a[finite], [1, 99])
    if hi <= lo:
        hi = lo + 1e-6
    return np.clip((a - lo) / (hi - lo), 0.0, 1.0)


def _resolve_dataset_mask_affine(dataset_name):
    ds = dataset_name.upper()

    # Preferred: reuse notebook's own transform helper used for Cellpose/ProSeg.
    if '_dataset_cellpose_transform' in globals():
        try:
            return _dataset_cellpose_transform(ds)
        except Exception as exc:
            print(f"[{dataset_name}] Warning: _dataset_cellpose_transform failed, falling back ({exc})")

    # Fallbacks if helper is unavailable.
    if ds == 'MERSCOPE':
        if 'MERSCOPE_TRANSFORM_PATH' in globals() and Path(MERSCOPE_TRANSFORM_PATH).exists():
            M = np.loadtxt(MERSCOPE_TRANSFORM_PATH)
            Minv = np.linalg.inv(M)
            return (
                (float(Minv[0, 0]), float(Minv[0, 1]), float(Minv[0, 2])),
                (float(Minv[1, 0]), float(Minv[1, 1]), float(Minv[1, 2])),
            )
        # Conservative fallback from known run logs.
        return (0.108, 0.0, 0.0), (0.0, 0.108, 0.0)

    if ds == 'XENIUM':
        mpp = None
        if 'XENIUM_SPEC_PATH' in globals() and Path(XENIUM_SPEC_PATH).exists():
            spec = json.loads(Path(XENIUM_SPEC_PATH).read_text())
            mpp = float(spec.get('pixel_size'))
        elif 'XENIUM_DIR' in globals() and Path(XENIUM_DIR).exists():
            candidate = Path(XENIUM_DIR) / 'experiment.xenium'
            if candidate.exists():
                spec = json.loads(candidate.read_text())
                mpp = float(spec.get('pixel_size'))
        if mpp is None:
            mpp = 0.2125
        return (float(mpp), 0.0, 0.0), (0.0, float(mpp), 0.0)

    raise ValueError(f'Unknown dataset: {dataset_name}')


def _affine_um_to_px(x_um, y_um, x_transform, y_transform):
    A = np.array([
        [float(x_transform[0]), float(x_transform[1])],
        [float(y_transform[0]), float(y_transform[1])],
    ], dtype=float)
    b = np.array([float(x_transform[2]), float(y_transform[2])], dtype=float)
    A_inv = np.linalg.inv(A)
    px, py = A_inv @ (np.array([x_um, y_um], dtype=float) - b)
    return float(px), float(py)


def _affine_px_to_um(x_px, y_px, x_transform, y_transform):
    x_um = float(x_transform[0]) * float(x_px) + float(x_transform[1]) * float(y_px) + float(x_transform[2])
    y_um = float(y_transform[0]) * float(x_px) + float(y_transform[1]) * float(y_px) + float(y_transform[2])
    return float(x_um), float(y_um)


def _get_background_image_crop(sdata_obj, dataset_name, bbox):
    if len(sdata_obj.images) == 0:
        return None

    if dataset_name.upper() == 'MERSCOPE':
        if 'MERSCOPE_z_projection' in sdata_obj.images:
            image_key = 'MERSCOPE_z_projection'
        else:
            image_key = next((k for k in sdata_obj.images.keys() if 'projection' in k.lower()), list(sdata_obj.images.keys())[0])
        ch2_pref = ['PolyT', '18S']
    else:
        image_key = 'morphology_focus' if 'morphology_focus' in sdata_obj.images else list(sdata_obj.images.keys())[0]
        ch2_pref = ['18S', 'PolyT']

    da = _get_scale0_dataarray(sdata_obj.images[image_key])
    x0_um, y0_um, x1_um, y1_um = bbox

    # Convert requested micron crop -> pixel crop for the image array.
    x_transform, y_transform = _resolve_dataset_mask_affine(dataset_name)
    corners_um = [
        (x0_um, y0_um),
        (x0_um, y1_um),
        (x1_um, y0_um),
        (x1_um, y1_um),
    ]
    corners_px = [_affine_um_to_px(xu, yu, x_transform, y_transform) for xu, yu in corners_um]
    px_vals = np.array([p[0] for p in corners_px], dtype=float)
    py_vals = np.array([p[1] for p in corners_px], dtype=float)

    # Pad by ~1 px to avoid edge truncation from float boundaries.
    px0, px1 = float(px_vals.min() - 1.0), float(px_vals.max() + 1.0)
    py0, py1 = float(py_vals.min() - 1.0), float(py_vals.max() + 1.0)

    crop = da.sel(x=slice(px0, px1), y=slice(py0, py1))
    if crop.sizes.get('x', 0) == 0 or crop.sizes.get('y', 0) == 0:
        return None

    channels = [str(c) for c in crop.coords['c'].values] if 'c' in crop.coords else []
    ch_dapi = _pick_channel_name(channels, ['DAPI']) if channels else None
    ch_rna = _pick_channel_name(channels, ch2_pref) if channels else None

    if 'c' in crop.dims:
        dapi_da = crop.sel(c=ch_dapi) if ch_dapi is not None else crop.isel(c=0)
        if ch_rna is not None:
            rna_da = crop.sel(c=ch_rna)
        else:
            rna_da = crop.isel(c=1 if crop.sizes['c'] > 1 else 0)
    else:
        dapi_da = crop
        rna_da = crop

    dapi = _norm01(np.asarray(dapi_da.compute() if hasattr(dapi_da, 'compute') else dapi_da))
    rna = _norm01(np.asarray(rna_da.compute() if hasattr(rna_da, 'compute') else rna_da))

    rgb = np.zeros((dapi.shape[0], dapi.shape[1], 3), dtype=np.float32)
    rgb[..., 2] = dapi  # DAPI in blue
    rgb[..., 1] = rna   # 18S / PolyT in green

    # Convert crop pixel bounds back to microns for plotting extent.
    xv = np.asarray(crop.coords['x'].values)
    yv = np.asarray(crop.coords['y'].values)
    px_corners = [
        (float(xv.min()), float(yv.min())),
        (float(xv.min()), float(yv.max())),
        (float(xv.max()), float(yv.min())),
        (float(xv.max()), float(yv.max())),
    ]
    um_corners = [_affine_px_to_um(px, py, x_transform, y_transform) for px, py in px_corners]
    x_um_vals = [u[0] for u in um_corners]
    y_um_vals = [u[1] for u in um_corners]
    extent_um = (min(x_um_vals), max(x_um_vals), min(y_um_vals), max(y_um_vals))

    return {
        'image_key': image_key,
        'channels_used': {'dapi': ch_dapi, 'rna_like': ch_rna},
        'rgb': rgb,
        'extent_um': extent_um,
        'transform': {'x_transform': x_transform, 'y_transform': y_transform},
    }


def plot_sanity_crop_panel(ax, sdata_obj, dataset_name, crop_size_um=250.0, center_xy=None, assignment_shape_key=SANITY_ASSIGNMENT_SHAPE_KEY):
    bbox, ref_shape_key = _choose_crop_bbox(sdata_obj, size_um=crop_size_um, center_xy=center_xy)
    x0, y0, x1, y1 = bbox

    # 1) Background image composite (micron-aware)
    bg = _get_background_image_crop(sdata_obj, dataset_name, bbox)
    if bg is not None:
        ax.imshow(bg['rgb'], extent=bg['extent_um'], origin='lower', interpolation='nearest', alpha=0.95)

    # 2) Overlay all shape layers with distinct colors
    shape_keys = list(sdata_obj.shapes.keys())
    cmap = plt.get_cmap('tab10')
    shape_handles = []
    shape_counts = {}

    for i, shape_key in enumerate(shape_keys):
        try:
            shp_crop = _crop_single_shape(sdata_obj, shape_key, bbox)
        except Exception as e:
            print(f"[{dataset_name}] Warning: failed to crop shapes[{shape_key}] ({e})")
            continue

        shape_counts[shape_key] = len(shp_crop)
        if len(shp_crop) == 0:
            continue

        color = cmap(i % 10)
        shp_crop.boundary.plot(ax=ax, linewidth=0.75, color=color, alpha=0.95)
        shape_handles.append(Line2D([0], [0], color=color, lw=2, label=f"{shape_key} ({len(shp_crop):,})"))

    # 3) Overlay transcripts colored by latest ProSeg assignment status
    tx_crop, points_key, assign_col = _crop_points(
        sdata_obj,
        bbox,
        max_points=SANITY_MAX_TRANSCRIPTS_PER_PANEL,
        random_state=SANITY_RANDOM_STATE,
        assignment_shape_key=assignment_shape_key,
    )

    tx_handles = []
    if len(tx_crop) > 0:
        unassigned = tx_crop[~tx_crop['assigned']]
        assigned = tx_crop[tx_crop['assigned']]

        if len(unassigned) > 0:
            ax.scatter(
                unassigned['x_um'], unassigned['y_um'],
                s=4, c='#d62728', alpha=0.50, rasterized=True
            )
            tx_handles.append(Line2D([0], [0], marker='o', linestyle='None', color='#d62728', label=f"Unassigned tx ({len(unassigned):,})", markersize=5))

        if len(assigned) > 0:
            ax.scatter(
                assigned['x_um'], assigned['y_um'],
                s=4, c='yellow', alpha=0.50, rasterized=True
            )
            tx_handles.append(Line2D([0], [0], marker='o', linestyle='None', color='yellow', label=f"Assigned tx ({len(assigned):,})", markersize=5))

    # Axes format (microns)
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect('equal')
    ax.set_xlabel('x (microns)')
    ax.set_ylabel('y (microns)')
    ax.set_title(f"{dataset_name} sanity crop ({crop_size_um:.0f} x {crop_size_um:.0f} um)")

    # Combined legend
    handles = tx_handles + shape_handles
    if handles:
        ax.legend(handles=handles, loc='upper right', frameon=True, fontsize=8)

    # Console summary
    n_assigned = int(tx_crop['assigned'].sum()) if len(tx_crop) else 0
    n_total = int(len(tx_crop))
    n_unassigned = n_total - n_assigned
    img_msg = 'none'
    if bg is not None:
        img_msg = (
            f"{bg['image_key']} (DAPI={bg['channels_used']['dapi']}, "
            f"RNA-like={bg['channels_used']['rna_like']})"
        )

    print(
        f"[{dataset_name}] bbox=({x0:.1f}, {y0:.1f})-({x1:.1f}, {y1:.1f}) | "
        f"ref_shape={ref_shape_key} | image={img_msg} | "
        f"points[{points_key}] tx={n_total:,} assigned={n_assigned:,} unassigned={n_unassigned:,} "
        f"assignment_col={assign_col}"
    )
    if bg is not None:
        print(
            f"    image transform x={bg['transform']['x_transform']} "
            f"y={bg['transform']['y_transform']}"
        )
    for sk in shape_keys:
        print(f"    shapes[{sk}] in crop: {shape_counts.get(sk, 0):,}")


status('[SANITY] Building 250 um crop overlays with micron-aware image alignment + all shape layers')
fig, axes = plt.subplots(1, 2, figsize=(17, 8), constrained_layout=True)
plot_sanity_crop_panel(axes[0], merscope_sdata, 'MERSCOPE', crop_size_um=SANITY_CROP_SIZE_UM, assignment_shape_key=SANITY_ASSIGNMENT_SHAPE_KEY)
plot_sanity_crop_panel(axes[1], xenium_sdata, 'XENIUM', crop_size_um=SANITY_CROP_SIZE_UM, assignment_shape_key=SANITY_ASSIGNMENT_SHAPE_KEY)
plt.show()


In [ ]:
# Whole-dataset transcript overview (3x2: density heatmaps + full scatter + fixed crop)
from matplotlib import colors as mcolors

TRANSCRIPT_OVERVIEW_SAMPLE_N = 250_000
TRANSCRIPT_OVERVIEW_CROP_SAMPLE_N = 200_000
TRANSCRIPT_OVERVIEW_RANDOM_STATE = 42
TRANSCRIPT_OVERVIEW_POINT_SIZE = 0.3
TRANSCRIPT_OVERVIEW_CROP_BBOX_UM = (4000.0, 4000.0, 6000.0, 6000.0)  # (x0, y0, x1, y1)
TRANSCRIPT_OVERVIEW_HEATMAP_BINS = 400
TRANSCRIPT_OVERVIEW_HEATMAP_CMAP = 'magma'
TRANSCRIPT_OVERVIEW_HEATMAP_LOG = False
TRANSCRIPT_OVERVIEW_HEATMAP_VMIN = None  # set numeric value to override auto min
TRANSCRIPT_OVERVIEW_HEATMAP_VMAX = 1700  # set numeric value to override auto max


def _first_existing_col_local2(df, candidates):
    cols = list(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None


def _sample_partition_local(df, n, rs):
    if len(df) <= n:
        return df
    return df.sample(n=n, random_state=rs)


def _resolve_points_xy_cols(sdata_obj):
    if len(sdata_obj.points) == 0:
        raise RuntimeError('No points found in SpatialData object.')
    pts_key = list(sdata_obj.points.keys())[0]
    pts = sdata_obj.points[pts_key]
    x_col = _first_existing_col_local2(pts, ['x', 'x_micron', 'global_x', 'x_location', 'observed_x'])
    y_col = _first_existing_col_local2(pts, ['y', 'y_micron', 'global_y', 'y_location', 'observed_y'])
    if x_col is None or y_col is None:
        raise KeyError(f'Could not resolve x/y columns for points[{pts_key}]')
    return pts_key, pts, x_col, y_col


def _points_bounds_and_sample_xy(sdata_obj, sample_n=250_000, random_state=42):
    pts_key, pts, x_col, y_col = _resolve_points_xy_cols(sdata_obj)
    work = pts[[x_col, y_col]]

    # Bounds via lazy reductions (cheap for very large datasets).
    if hasattr(work, 'npartitions') and hasattr(work, 'compute'):
        minx = float(work[x_col].min().compute())
        maxx = float(work[x_col].max().compute())
        miny = float(work[y_col].min().compute())
        maxy = float(work[y_col].max().compute())

        total_n = int(work.map_partitions(len, meta=('n', 'i8')).sum().compute())
        if total_n <= sample_n:
            sampled = work.compute()
        else:
            # Unbiased sampling: draw by global fraction, not equal-per-partition.
            frac = float(sample_n) / float(total_n)
            sampled = work.sample(frac=frac, random_state=random_state).compute()
        sampled = sampled.rename(columns={x_col: 'x_um', y_col: 'y_um'})
    else:
        pdf = pd.DataFrame(work).copy()
        minx = float(pdf[x_col].min())
        maxx = float(pdf[x_col].max())
        miny = float(pdf[y_col].min())
        maxy = float(pdf[y_col].max())
        sampled = pdf.rename(columns={x_col: 'x_um', y_col: 'y_um'})

    if len(sampled) > sample_n:
        sampled = sampled.sample(n=sample_n, random_state=random_state)

    return {
        'points_key': pts_key,
        'sampled': sampled,
        'bounds': (minx, miny, maxx, maxy),
    }


def _points_sample_xy_in_bbox(sdata_obj, bbox, sample_n=200_000, random_state=42):
    x0, y0, x1, y1 = bbox
    pts_key, pts, x_col, y_col = _resolve_points_xy_cols(sdata_obj)

    work = pts[[x_col, y_col]]

    if hasattr(work, 'npartitions') and hasattr(work, 'compute'):
        crop = work[
            (work[x_col] >= x0) & (work[x_col] <= x1) &
            (work[y_col] >= y0) & (work[y_col] <= y1)
        ]
        total_n = int(crop.map_partitions(len, meta=('n', 'i8')).sum().compute())
        if total_n <= sample_n:
            sampled = crop.compute()
        else:
            frac = float(sample_n) / float(total_n)
            sampled = crop.sample(frac=frac, random_state=random_state).compute()
        sampled = sampled.rename(columns={x_col: 'x_um', y_col: 'y_um'})
    else:
        pdf = pd.DataFrame(work).copy()
        sampled = pdf[
            (pdf[x_col] >= x0) & (pdf[x_col] <= x1) &
            (pdf[y_col] >= y0) & (pdf[y_col] <= y1)
        ].copy()
        sampled = sampled.rename(columns={x_col: 'x_um', y_col: 'y_um'})

    if len(sampled) > sample_n:
        sampled = sampled.sample(n=sample_n, random_state=random_state)

    return {
        'points_key': pts_key,
        'sampled': sampled,
        'bbox': bbox,
    }


def _hist2d_all_points_streaming(sdata_obj, x_range, y_range, bins=400, dataset_name='DATASET'):
    """Compute 2D transcript density histogram over all transcripts with partition-wise streaming."""
    pts_key, pts, x_col, y_col = _resolve_points_xy_cols(sdata_obj)
    work = pts[[x_col, y_col]]

    hist = np.zeros((bins, bins), dtype=np.uint64)

    if hasattr(work, 'npartitions') and hasattr(work, 'get_partition') and hasattr(work, 'compute'):
        nparts = int(work.npartitions)
        iterator = tqdm(range(nparts), desc=f'[{dataset_name}] density histogram partitions', unit='part')
        for i in iterator:
            pdf = work.get_partition(i).compute()
            if len(pdf) == 0:
                continue
            xv = pdf[x_col].to_numpy(dtype=np.float64, copy=False)
            yv = pdf[y_col].to_numpy(dtype=np.float64, copy=False)
            h, _, _ = np.histogram2d(xv, yv, bins=bins, range=[x_range, y_range])
            hist += h.astype(np.uint64, copy=False)
    else:
        pdf = pd.DataFrame(work).copy()
        if len(pdf) > 0:
            xv = pdf[x_col].to_numpy(dtype=np.float64, copy=False)
            yv = pdf[y_col].to_numpy(dtype=np.float64, copy=False)
            h, _, _ = np.histogram2d(xv, yv, bins=bins, range=[x_range, y_range])
            hist += h.astype(np.uint64, copy=False)

    return {
        'points_key': pts_key,
        'hist': hist,
        'n_total': int(hist.sum()),
        'x_col': x_col,
        'y_col': y_col,
    }


status('[SANITY] Building 3x2 transcript overview (density + full + fixed crop)')

if 'merscope_sdata' not in globals() or 'xenium_sdata' not in globals():
    raise NameError('Expected merscope_sdata and xenium_sdata in memory. Run the loading cell first.')

# Full-field sampled clouds
merscope_full = _points_bounds_and_sample_xy(
    merscope_sdata,
    sample_n=TRANSCRIPT_OVERVIEW_SAMPLE_N,
    random_state=TRANSCRIPT_OVERVIEW_RANDOM_STATE,
)
xenium_full = _points_bounds_and_sample_xy(
    xenium_sdata,
    sample_n=TRANSCRIPT_OVERVIEW_SAMPLE_N,
    random_state=TRANSCRIPT_OVERVIEW_RANDOM_STATE,
)

# Fixed micron crop sampled clouds
crop_bbox = TRANSCRIPT_OVERVIEW_CROP_BBOX_UM
merscope_crop = _points_sample_xy_in_bbox(
    merscope_sdata,
    bbox=crop_bbox,
    sample_n=TRANSCRIPT_OVERVIEW_CROP_SAMPLE_N,
    random_state=TRANSCRIPT_OVERVIEW_RANDOM_STATE,
)
xenium_crop = _points_sample_xy_in_bbox(
    xenium_sdata,
    bbox=crop_bbox,
    sample_n=TRANSCRIPT_OVERVIEW_CROP_SAMPLE_N,
    random_state=TRANSCRIPT_OVERVIEW_RANDOM_STATE,
)

# Match micron scale across top/middle rows.
def _center_and_span(bounds):
    minx, miny, maxx, maxy = bounds
    cx = 0.5 * (minx + maxx)
    cy = 0.5 * (miny + maxy)
    span = max(maxx - minx, maxy - miny)
    return cx, cy, span

m_cx, m_cy, m_span = _center_and_span(merscope_full['bounds'])
x_cx, x_cy, x_span = _center_and_span(xenium_full['bounds'])
common_span = max(m_span, x_span)
half = 0.5 * common_span

m_xrange = (m_cx - half, m_cx + half)
m_yrange = (m_cy - half, m_cy + half)
x_xrange = (x_cx - half, x_cx + half)
x_yrange = (x_cy - half, x_cy + half)

# Top row: all-transcript density heatmaps (shared colormap + limits)
status('[SANITY] Computing all-transcript density heatmap for MERSCOPE')
m_heat = _hist2d_all_points_streaming(
    merscope_sdata,
    x_range=m_xrange,
    y_range=m_yrange,
    bins=TRANSCRIPT_OVERVIEW_HEATMAP_BINS,
    dataset_name='MERSCOPE',
)
status('[SANITY] Computing all-transcript density heatmap for XENIUM')
x_heat = _hist2d_all_points_streaming(
    xenium_sdata,
    x_range=x_xrange,
    y_range=x_yrange,
    bins=TRANSCRIPT_OVERVIEW_HEATMAP_BINS,
    dataset_name='XENIUM',
)

m_h = m_heat['hist'].T  # (y, x) for imshow
x_h = x_heat['hist'].T

shared_vmax_auto = float(max(np.nanmax(m_h), np.nanmax(x_h), 1.0))
positive_mins = []
if np.any(m_h > 0):
    positive_mins.append(float(np.nanmin(m_h[m_h > 0])))
if np.any(x_h > 0):
    positive_mins.append(float(np.nanmin(x_h[x_h > 0])))
shared_vmin_positive_auto = float(min(positive_mins)) if positive_mins else 1.0

if TRANSCRIPT_OVERVIEW_HEATMAP_LOG:
    auto_vmin = shared_vmin_positive_auto
    auto_vmax = shared_vmax_auto
else:
    auto_vmin = 0.0
    auto_vmax = shared_vmax_auto

heat_vmin = float(TRANSCRIPT_OVERVIEW_HEATMAP_VMIN) if TRANSCRIPT_OVERVIEW_HEATMAP_VMIN is not None else auto_vmin
heat_vmax = float(TRANSCRIPT_OVERVIEW_HEATMAP_VMAX) if TRANSCRIPT_OVERVIEW_HEATMAP_VMAX is not None else auto_vmax

if TRANSCRIPT_OVERVIEW_HEATMAP_LOG and heat_vmin <= 0:
    raise ValueError('For log heatmap scale, TRANSCRIPT_OVERVIEW_HEATMAP_VMIN must be > 0')
if heat_vmax <= heat_vmin:
    raise ValueError(
        f'Invalid heatmap limits: vmin={heat_vmin} must be < vmax={heat_vmax}. '
        'Adjust TRANSCRIPT_OVERVIEW_HEATMAP_VMIN/VMAX.'
    )

if TRANSCRIPT_OVERVIEW_HEATMAP_LOG:
    norm = mcolors.LogNorm(vmin=heat_vmin, vmax=heat_vmax)
    m_show = np.where(m_h > 0, m_h, np.nan)
    x_show = np.where(x_h > 0, x_h, np.nan)
else:
    norm = mcolors.Normalize(vmin=heat_vmin, vmax=heat_vmax)
    m_show = m_h
    x_show = x_h

x0, y0, x1, y1 = crop_bbox

fig, axes = plt.subplots(3, 2, figsize=(16, 20), constrained_layout=True)

# Row 1: heatmaps
im0 = axes[0, 0].imshow(
    m_show,
    origin='lower',
    extent=(m_xrange[0], m_xrange[1], m_yrange[0], m_yrange[1]),
    cmap=TRANSCRIPT_OVERVIEW_HEATMAP_CMAP,
    norm=norm,
    aspect='equal',
)
axes[0, 0].set_title('MERSCOPE transcript density (all transcripts)')
axes[0, 0].set_xlabel('x (microns)')
axes[0, 0].set_ylabel('y (microns)')

im1 = axes[0, 1].imshow(
    x_show,
    origin='lower',
    extent=(x_xrange[0], x_xrange[1], x_yrange[0], x_yrange[1]),
    cmap=TRANSCRIPT_OVERVIEW_HEATMAP_CMAP,
    norm=norm,
    aspect='equal',
)
axes[0, 1].set_title('XENIUM transcript density (all transcripts)')
axes[0, 1].set_xlabel('x (microns)')
axes[0, 1].set_ylabel('y (microns)')

cbar = fig.colorbar(im1, ax=[axes[0, 0], axes[0, 1]], shrink=0.9)
cbar.set_label('Transcript count per bin' + (' (log scale)' if TRANSCRIPT_OVERVIEW_HEATMAP_LOG else ''))

# Row 2: full-field scatter subsample
m_df = merscope_full['sampled']
axes[1, 0].scatter(m_df['x_um'], m_df['y_um'], s=TRANSCRIPT_OVERVIEW_POINT_SIZE, c='#1f77b4', alpha=0.15, rasterized=True)
axes[1, 0].set_xlim(m_xrange)
axes[1, 0].set_ylim(m_yrange)
axes[1, 0].set_aspect('equal')
axes[1, 0].set_title('MERSCOPE transcripts (subsample, full)')
axes[1, 0].set_xlabel('x (microns)')
axes[1, 0].set_ylabel('y (microns)')

x_df = xenium_full['sampled']
axes[1, 1].scatter(x_df['x_um'], x_df['y_um'], s=TRANSCRIPT_OVERVIEW_POINT_SIZE, c='#d62728', alpha=0.15, rasterized=True)
axes[1, 1].set_xlim(x_xrange)
axes[1, 1].set_ylim(x_yrange)
axes[1, 1].set_aspect('equal')
axes[1, 1].set_title('XENIUM transcripts (subsample, full)')
axes[1, 1].set_xlabel('x (microns)')
axes[1, 1].set_ylabel('y (microns)')

# Row 3: fixed crop scatter
mc_df = merscope_crop['sampled']
axes[2, 0].scatter(mc_df['x_um'], mc_df['y_um'], s=TRANSCRIPT_OVERVIEW_POINT_SIZE, c='#1f77b4', alpha=0.05, rasterized=True)
axes[2, 0].set_xlim(x0, x1)
axes[2, 0].set_ylim(y0, y1)
axes[2, 0].set_aspect('equal')
axes[2, 0].set_title(f'MERSCOPE crop x=[{x0:.0f},{x1:.0f}], y=[{y0:.0f},{y1:.0f}]')
axes[2, 0].set_xlabel('x (microns)')
axes[2, 0].set_ylabel('y (microns)')

xc_df = xenium_crop['sampled']
axes[2, 1].scatter(xc_df['x_um'], xc_df['y_um'], s=TRANSCRIPT_OVERVIEW_POINT_SIZE, c='#d62728', alpha=0.05, rasterized=True)
axes[2, 1].set_xlim(x0, x1)
axes[2, 1].set_ylim(y0, y1)
axes[2, 1].set_aspect('equal')
axes[2, 1].set_title(f'XENIUM crop x=[{x0:.0f},{x1:.0f}], y=[{y0:.0f},{y1:.0f}]')
axes[2, 1].set_xlabel('x (microns)')
axes[2, 1].set_ylabel('y (microns)')

print(f"MERSCOPE density (all): n={m_heat['n_total']:,} | bins={TRANSCRIPT_OVERVIEW_HEATMAP_BINS} | points key={m_heat['points_key']}")
print(f"XENIUM density (all): n={x_heat['n_total']:,} | bins={TRANSCRIPT_OVERVIEW_HEATMAP_BINS} | points key={x_heat['points_key']}")
print(f"MERSCOPE full scatter sampled={len(m_df):,} | bounds={merscope_full['bounds']}")
print(f"XENIUM full scatter sampled={len(x_df):,} | bounds={xenium_full['bounds']}")
print(f"MERSCOPE crop sampled={len(mc_df):,} | bbox={crop_bbox}")
print(f"XENIUM crop sampled={len(xc_df):,} | bbox={crop_bbox}")
print(f"Heatmap scale={'log' if TRANSCRIPT_OVERVIEW_HEATMAP_LOG else 'linear'} | vmin={heat_vmin:,.3g} | vmax={heat_vmax:,.3g}")
print(f"Top/middle common span={common_span:,.1f} microns")

plt.show()
